# nb_FactClaims
## Carga de la tabla Gold.bgla_FactClaim
**Patrón de carga:** Fact (Bronze → Silver staging → Gold)

**Fuente:** Tablas Bronze de AmigosPlus (Claim_Detail, Claim_LineDetail, Claim_Log, etc.)

**Descripción:** Migración del SP `usp_DataPrepare_FactClaim` a Fabric Notebook.
Genera registros de claims desde múltiples fuentes con resolución de dimensiones,
cálculos de montos en múltiples monedas, flags de negocio (ICU, Emergency, Readmissions),
y procesos post-carga (DiseasesAssociated, Readmissions, Hospital_Emergency).

**Convenciones BGLA:**
- Staging: `Silver.TmpFactClaim`
- Target: `Gold.bgla_FactClaim`
- Defaults: Strings → `'_Undefined'`, Numbers → `0` / `-101`, Dates → `'1899-12-31'`

In [2]:
%%sql
DELETE FROM  Gold.bgla_FactClaim

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
# ============================================================
# SPARK PERFORMANCE CONFIGURATION
# Buenas prácticas Fabric: AQE, Optimize Write, V-Order
# ============================================================

# ── Adaptive Query Execution (AQE) ──
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# ── Delta Write Optimization ──
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
spark.conf.set("spark.microsoft.delta.parquet.vorder.enabled", "true")

# ── Shuffle Partitions (ajustar según volumen de datos) ──
spark.conf.set("spark.sql.shuffle.partitions", "200")

# ── Broadcast threshold (tablas < 100MB se hacen broadcast automáticamente) ──
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")  # 100MB

# ── Schema auto-merge para Delta ──
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ── Legacy parquet datetime ──
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInWrite", "CORRECTED")

print("[OK] Spark performance configuration applied")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 9, Finished, Available, Finished, False)

[OK] Spark performance configuration applied


In [8]:
# ============================================================
# PARÁMETROS DE CARGA
# Equivale a @DummyLargo, @Min, @Max del SP
# ============================================================

DUMMY_LARGO = '_Undefined'

# ── Outlier config ──
MIN_OUTLIER = 0
MAX_OUTLIER = 99999999

try:
    df_config = spark.sql("""
        SELECT Propiedad, CAST(Valor AS INT) AS Valor
        FROM Bronze.ETLFramework_Metadata_TConfiguracion
        WHERE FiltroConfiguracion = 'Claim Outlier'
          AND Propiedad IN ('Min Outlier Billed Amount', 'Max Outlier Billed Amount')
    """)
    config_map = {row["Propiedad"]: row["Valor"] for row in df_config.collect()}
    MIN_OUTLIER = config_map.get("Min Outlier Billed Amount", MIN_OUTLIER)
    MAX_OUTLIER = config_map.get("Max Outlier Billed Amount", MAX_OUTLIER)
    print(f"[OK] Outlier config from Bronze: Min={MIN_OUTLIER}, Max={MAX_OUTLIER}")
except Exception:
    print(f"[WARN] TConfiguracion not available, using defaults: Min={MIN_OUTLIER}, Max={MAX_OUTLIER}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 10, Finished, Available, Finished, False)

[WARN] TConfiguracion not available, using defaults: Min=0, Max=99999999


In [9]:
# ============================================================
# Limpiar staging table (se mantiene la estructura)
# ============================================================
if spark.catalog.tableExists("Silver.TmpFactClaim"):
    spark.sql("DELETE FROM Silver.TmpFactClaim")
    print("[OK] Silver.TmpFactClaim truncated (rows deleted)")
else:
    print("[INFO] Silver.TmpFactClaim does not exist yet — will be created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 11, Finished, Available, Finished, False)

[OK] Silver.TmpFactClaim truncated (rows deleted)


In [10]:
# ============================================================
# DELTA SETUP: vw_TClaimDetail (= #TClaimDetail del SP)
# Watermark basado en MAX(LogId) de Gold.bgla_FactClaim
# ============================================================
from pyspark.sql.utils import AnalysisException

# ── FILTRO DE FECHA (opcional) ─────────────────────────────
# Establecer en None para usar el delta completo por LogId.
# Establecer fechas (YYYY-MM-DD) para limitar la carga a un rango por ServiceFromDate (cld.FromDate).
# Útil para reprocesar un solo mes y acelerar la carga.
FILTER_FROM_DATE = None  # ej: '2026-04-01'  (None = sin filtro)
FILTER_TO_DATE   = None   # ej: '2026-04-30'  (None = sin filtro)
USE_LOGID_WATERMARK = True       # False = ignora watermark y carga TODO el rango de fechas (recomendado para reproceso de un mes)
# ───────────────────────────────────────────────────────────

# Paso 1: asegurar columna LogId en Gold
try:
    spark.sql("ALTER TABLE Gold.bgla_FactClaim ADD COLUMNS (LogId BIGINT)")
    print("[INFO] Columna LogId agregada a Gold.bgla_FactClaim")
except AnalysisException as e:
    if "already exists" in str(e).lower() or "duplicate" in str(e).lower():
        print("[OK] Columna LogId ya existe en Gold.bgla_FactClaim")
    else:
        raise

# Paso 2: watermark
if USE_LOGID_WATERMARK:
    max_logid = (
        spark.sql("SELECT COALESCE(MAX(LogId), 0) AS max_logid FROM Gold.bgla_FactClaim")
             .collect()[0]["max_logid"] or 0
    )
    print(f"[OK] Watermark delta (max LogId): {max_logid}")
else:
    max_logid = 0
    print("[INFO] Watermark LogId DESHABILITADO — se cargará todo el rango de fechas indicado")

# Paso 3: construir vw_TClaimDetail con filtros opcionales
where_parts = [f"cl.LogId >= {max_logid}"]
if FILTER_FROM_DATE:
    where_parts.append(f"cld.FromDate >= DATE '{FILTER_FROM_DATE}'")
if FILTER_TO_DATE:
    where_parts.append(f"cld.FromDate <= DATE '{FILTER_TO_DATE}'")
where_clause = ' AND '.join(where_parts)

df_delta = spark.sql(f"""
    SELECT cd.HeaderId, MAX(cl.LogId) AS LogId
    FROM Bronze.AmigosPlus_AMP_Claim_Log cl
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cl.ClaimDetailId = cd.ClaimDetailId
    INNER JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cd.ClaimDetailId = cld.ClaimDetailId
    WHERE {where_clause}
    GROUP BY cd.HeaderId
""")
df_delta.cache()
df_delta.createOrReplaceTempView("vw_TClaimDetail")
delta_count = df_delta.count()
print(f"[OK] vw_TClaimDetail cached — {delta_count:,} HeaderIds to process")
if FILTER_FROM_DATE or FILTER_TO_DATE:
    print(f"[INFO] Filtro activo por ServiceFromDate: {FILTER_FROM_DATE or '-inf'} → {FILTER_TO_DATE or '+inf'}")


StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 12, Finished, Available, Finished, False)

[OK] Columna LogId ya existe en Gold.bgla_FactClaim
[OK] Watermark delta (max LogId): 0
[OK] vw_TClaimDetail cached — 6,223,219 HeaderIds to process


In [11]:
# Vista temporal: ClaimLog (último log activo por ClaimDetail)
df_claim_log = spark.sql("""
    SELECT cl.ClaimDetailId, cl.StatusId, cl.ReasonId, cl.FromDate, cl.UpdatedBy,
           ROW_NUMBER() OVER (PARTITION BY cl.ClaimDetailId ORDER BY cl.LogId DESC) AS rn
    FROM Bronze.AmigosPlus_AMP_Claim_Log cl
    WHERE cl.ToDate IS NULL
""")
df_claim_log.createOrReplaceTempView("vw_ClaimLog")
print("[OK] vw_ClaimLog created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 13, Finished, Available, Finished, False)

[OK] vw_ClaimLog created


In [12]:
# Vista temporal: CurrentVersion (versión vigente del claim)
df_current_version = spark.sql("""
    SELECT DISTINCT cd.HeaderId, cd.ClaimDetailId
    FROM Bronze.AmigosPlus_AMP_Claim_Log cl
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON cl.ClaimDetailId = cd.ClaimDetailId
    WHERE cl.ToDate IS NULL
""")
df_current_version.createOrReplaceTempView("vw_CurrentVersion")
print("[OK] vw_CurrentVersion created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 14, Finished, Available, Finished, False)

[OK] vw_CurrentVersion created


In [13]:
# Vista temporal: ProcessedDate (fecha de procesamiento del claim)
df_processed_date = spark.sql("""
    SELECT l.ClaimDetailId, l.StatusId, l.ReasonId, l.FromDate,
           ROW_NUMBER() OVER (PARTITION BY l.ClaimDetailId ORDER BY l.LogId DESC) AS Cont
    FROM Bronze.AmigosPlus_AMP_Claim_Log l
    WHERE l.StatusId = 1017
      AND l.ReasonId IN (1094, 1095, 1096, 1773, 1827, 1828, 1829, 1933)
""")
df_processed_date.createOrReplaceTempView("vw_ProcessedDate")
print("[OK] vw_ProcessedDate created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 15, Finished, Available, Finished, False)

[OK] vw_ProcessedDate created


In [14]:
# Vista temporal: DiagnosticLine (diagnóstico primario por línea)
df_diagnostic_line = spark.sql("""
    SELECT diag.DiagnosticId, diag.ClaimLineDetailId,
           ROW_NUMBER() OVER (PARTITION BY diag.ClaimLineDetailId ORDER BY diag.DiagnosticLineId DESC) AS ID
    FROM Bronze.AmigosPlus_AMP_Claim_DiagnosticbyLine diag
    WHERE diag.`Primary` = 1
""")
df_diagnostic_line.createOrReplaceTempView("vw_DiagnosticLine")
print("[OK] vw_DiagnosticLine created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 16, Finished, Available, Finished, False)

[OK] vw_DiagnosticLine created


In [15]:
# Vista temporal: ProcedureLine (procedimiento primario por línea)
df_procedure_line = spark.sql("""
    SELECT procl.ProcedureId, procl.ClaimLineDetailId,
           ROW_NUMBER() OVER (PARTITION BY procl.ClaimLineDetailId ORDER BY procl.ProcedureByLineId DESC) AS ID
    FROM Bronze.AmigosPlus_AMP_Claim_ProcedurebyLine procl
    WHERE procl.`Primary` = 1
""")
df_procedure_line.createOrReplaceTempView("vw_ProcedureLine")
print("[OK] vw_ProcedureLine created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 17, Finished, Available, Finished, False)

[OK] vw_ProcedureLine created


In [16]:
# Vista temporal: ClaimSubmission (último envío digital por claim)
df_claim_submission = spark.sql("""
    SELECT DCS.ClaimHeaderId, DCS.PolicyId, DCS.SenderTypeId, DCS.InboundChannel,
           DCS.TrackingNumber, DCS.Updatedon, DCS.ReceivedDate, DCS.IsComplement,
           DCS.SenderEmailAddress, DCS.SenderFullName,
           ROW_NUMBER() OVER (PARTITION BY DCS.ClaimHeaderId ORDER BY DCS.updatedon DESC) AS CONT
    FROM Bronze.Claim_ClaimSubmission DCS
    WHERE DCS.ClaimHeaderId <> 0
""")
df_claim_submission.createOrReplaceTempView("vw_ClaimSubmission")
print("[OK] vw_ClaimSubmission created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 18, Finished, Available, Finished, False)

[OK] vw_ClaimSubmission created


In [17]:
# Vista temporal: TmpReceived (primera fecha de recepción por header, cont=1)
df_tmp_received = spark.sql("""
    SELECT cd.HeaderId, cd.ClaimDetailId, cd.ReceivedDate,
           ROW_NUMBER() OVER (PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId) AS Cont
    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
""")
df_tmp_received.createOrReplaceTempView("vw_TmpReceived")
print("[OK] vw_TmpReceived created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 19, Finished, Available, Finished, False)

[OK] vw_TmpReceived created


In [18]:
# Vista temporal: TmpServices (primera fecha de servicio por header)
df_tmp_services = spark.sql("""
    SELECT cd.HeaderId, MIN(cld.FromDate) AS ServiceFromDate
    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
    INNER JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cld.ClaimDetailId = cd.ClaimDetailId
    GROUP BY cd.HeaderId
""")
df_tmp_services.createOrReplaceTempView("vw_TmpServices")
print("[OK] vw_TmpServices created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 20, Finished, Available, Finished, False)

[OK] vw_TmpServices created


In [19]:
# Vista temporal: PolicyBaseCurrency (moneda base por póliza)
df_policy_base_currency = spark.sql("""
    SELECT DISTINCT me.PolicyId, me.MemberId, me.MemberEligibilityId, me.FromDate, me.ToDate,
           cr.CurrencyId AS BaseCurrencyId
    FROM Bronze.AmigosPlus_AMP_Policy_MemberEligibility me
    INNER JOIN Bronze.AmigosPlus_AMP_Common_DeductibleRegionPlanXRef drpx ON me.DeductibleRegionPlanXRefId = drpx.DeductibleRegionPlanXRefId
    INNER JOIN Bronze.AmigosPlus_AMP_Common_Plan pl ON drpx.PlanId = pl.PlanId
    INNER JOIN Bronze.AmigosPlus_AMP_Common_Product pr ON pl.ProductId = pr.ProductId
    LEFT JOIN Bronze.AmigosPlus_AMP_Generic_CurrencyRegionProductXRef crpx ON pr.ProductId = crpx.ProductId AND drpx.RegionId = crpx.RegionId
    INNER JOIN Bronze.AmigosPlus_AMP_Generic_Currency cr ON COALESCE(crpx.CurrencyId, pr.BaseCurrencyId) = cr.CurrencyId
    WHERE me.RelationTypeId IN (2, 5) AND me.StatusId <> 30 AND me.FromDate <= COALESCE(me.ToDate, me.FromDate)
""")
df_policy_base_currency.createOrReplaceTempView("vw_PolicyBaseCurrency")
print("[OK] vw_PolicyBaseCurrency created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 21, Finished, Available, Finished, False)

[OK] vw_PolicyBaseCurrency created


In [20]:
# Vista temporal: Payee (payee preferido por contacto)
df_payee = spark.sql("""
    SELECT PayeeId, ContactBaseId,
           ROW_NUMBER() OVER (PARTITION BY ContactBaseId ORDER BY PayeeId DESC) AS RowNum
    FROM Bronze.AmigosPlus_AMP_Common_Payee
    WHERE ToDate IS NULL AND (ClaimPreferred = 1 OR ClaimPreferredOnShore = 1)
""")
df_payee.createOrReplaceTempView("vw_Payee")
print("[OK] vw_Payee created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 22, Finished, Available, Finished, False)

[OK] vw_Payee created


In [21]:
# Vista temporal: ReceivedMethodDigital (métodos de recepción digitales)
df_received_method_digital = spark.sql("""
    SELECT ReceivedMethodId, ReceivedMethodName
    FROM (
        SELECT rm.ReceivedMethodId, rm.Name AS ReceivedMethodName
        FROM Bronze.AmigosPlus_AMP_Common_ReceivedMethod rm
        UNION
        SELECT cc.CodeId AS ReceivedMethodId, cc.Name AS ReceivedMethodName
        FROM Bronze.AmigosPlus_AMP_Common_Code cc
        WHERE cc.CodeTypeId = 3109
        UNION
        SELECT -102 AS ReceivedMethodId, 'Web' AS ReceivedMethodName
    ) allMethods
    WHERE ReceivedMethodName IN ('Web-Single', 'Web-Multiple', 'Web')
""")
df_received_method_digital.createOrReplaceTempView("vw_ReceivedMethodDigital")
print("[OK] vw_ReceivedMethodDigital created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 23, Finished, Available, Finished, False)

[OK] vw_ReceivedMethodDigital created


In [22]:
# ============================================================
# Vista temporal: LocalTeamTAT (IsCOR y LocalTeam desde FactClaimTAT)
# Equivale a #MaxIteratiopn + #LocalTeamTAT del SP
# ============================================================
if spark.catalog.tableExists("Gold.bgla_FactClaimTAT"):
    df_local_team_tat = spark.sql("""
        SELECT DISTINCT tat.LocalTeam, tat.ClaimHeaderId, tat.IsCOR
        FROM Gold.bgla_FactClaimTAT tat
        INNER JOIN (
            SELECT ClaimHeaderId, MAX(Iterations) AS MaxIteration
            FROM Gold.bgla_FactClaimTAT
            WHERE ClaimHeaderId IN (SELECT DISTINCT HeaderId FROM vw_TClaimDetail)
            GROUP BY ClaimHeaderId
        ) mi ON mi.ClaimHeaderId = tat.ClaimHeaderId AND mi.MaxIteration = tat.Iterations
        WHERE tat.ClaimHeaderId IN (SELECT DISTINCT HeaderId FROM vw_TClaimDetail)
    """)
    df_local_team_tat.createOrReplaceTempView("vw_LocalTeamTAT")
    print(f"[OK] vw_LocalTeamTAT created — {df_local_team_tat.count():,} rows")
else:
    spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS ClaimHeaderId,
               CAST('_Non Defined IsCOR' AS STRING) AS IsCOR,
               CAST('_Non Defined LocalTeam' AS STRING) AS LocalTeam
        WHERE 1 = 0
    """).createOrReplaceTempView("vw_LocalTeamTAT")
    print("[WARN] Gold.bgla_FactClaimTAT not found — vw_LocalTeamTAT created as empty stub")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 24, Finished, Available, Finished, False)

[WARN] Gold.bgla_FactClaimTAT not found — vw_LocalTeamTAT created as empty stub


In [23]:
# ============================================================
# Vista temporal: HighCostClaim (claims de alto costo por medicación)
# Equivale a #claimHighCost del SP — PDOCM en TAT o ValidationLogStep=5
# ============================================================
try:
    df_high_cost = spark.sql("""
        SELECT DISTINCT ClaimHeaderId
        FROM (
            SELECT DISTINCT TAT.ClaimHeaderId
            FROM Gold.bgla_FactClaimTAT TAT
            INNER JOIN vw_TClaimDetail CD ON TAT.ClaimHeaderId = CD.HeaderId
            WHERE TAT.Iterations <> 0 AND TAT.Reason = 'PDOCM'
            UNION
            SELECT DISTINCT cd.HeaderId AS ClaimHeaderId
            FROM vw_TClaimDetail CDl
            INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON CDl.HeaderId = cd.HeaderId
            INNER JOIN Bronze.AmigosPlus_AMP_Claim_ValidationLogClaim clv ON clv.ClaimDetailId = cd.ClaimDetailId
            WHERE clv.ValidationLogStepId = 5
        ) T
    """)
    df_high_cost.createOrReplaceTempView("vw_HighCostClaim")
    print(f"[OK] vw_HighCostClaim created — {df_high_cost.count():,} rows")
except Exception as e:
    spark.sql("SELECT CAST(-1 AS BIGINT) AS ClaimHeaderId WHERE 1 = 0").createOrReplaceTempView("vw_HighCostClaim")
    print(f"[WARN] HighCostClaim dependencies not available — using empty stub: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 25, Finished, Available, Finished, False)

[WARN] HighCostClaim dependencies not available — using empty stub: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Gold`.`bgla_FactClaimTAT` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 5 pos 17;
'Distinct
+- 'Project ['ClaimHeaderId]
   +- 'SubqueryAlias T
      +- 'Distinct
         +- 'Union false, false
            :- 'Distinct
            :  +- 'Project ['TAT.ClaimHeaderId]
            :     +- 'Filter (NOT ('TAT.Iterations = 0) AND ('TAT.Reason = PDOCM))
            :        +- 'Join Inner, ('TAT.ClaimHeaderId = 'CD.HeaderId)
            :           :- 'SubqueryAlias TAT
            :           :  +- 'UnresolvedRelation [Gold, bgla_FactClaimTAT], [], false
            :           +- SubqueryAlias CD
            :              

In [24]:
# ============================================================
# Vista temporal: CoinsurancePercent (BISI Coinsurance)
# Equivale a #BaseCoinsurancePercent + #CoinsurancePercent del SP
# ============================================================
try:
    df_coinsurance = spark.sql("""
        SELECT cv.ClaimDetailId, cv.ClaimLineDetailId,
               COALESCE(
                   GET_JSON_OBJECT(cv.ValidationDetails, '$[0].coveragebyPlanResponse[0].Coinsurance'),
                   GET_JSON_OBJECT(cv.ValidationDetails, '$.CoverageDetails[0].Coinsurance'),
                   '0'
               ) AS CoinsurancePercent
        FROM Bronze.AmigosPlus_AMP_Claim_ValidationLogClaim cv
        INNER JOIN vw_TClaimDetail CD ON cv.ClaimID = CD.HeaderId
        WHERE cv.ValidationLogStepId = 1
          AND INSTR(CAST(cv.ValidationDetails AS STRING), 'Coinsurance') > 0
          AND COALESCE(
                  GET_JSON_OBJECT(cv.ValidationDetails, '$[0].coveragebyPlanResponse[0].Coinsurance'),
                  GET_JSON_OBJECT(cv.ValidationDetails, '$.CoverageDetails[0].Coinsurance')
              ) IS NOT NULL
    """)
    df_coinsurance.createOrReplaceTempView("vw_CoinsurancePercent")
    print(f"[OK] vw_CoinsurancePercent created — {df_coinsurance.count():,} rows")
except Exception as e:
    spark.sql("""
        SELECT CAST(-1 AS INT) AS ClaimDetailId, CAST(-1 AS BIGINT) AS ClaimLineDetailId,
               CAST('0' AS STRING) AS CoinsurancePercent WHERE 1 = 0
    """).createOrReplaceTempView("vw_CoinsurancePercent")
    print(f"[WARN] ValidationLogClaim not available — using empty stub: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 26, Finished, Available, Finished, False)

[OK] vw_CoinsurancePercent created — 15,106 rows


In [25]:
# ============================================================
# Vista temporal: ClearingHouse (datos de ClearingHouse por header)
# Equivale a #ClearingHouse del SP
# Fuente: Bronze.Clearinghouse_Invoice_* (migrado de ETLStaging.Clearinghouse)
# ============================================================
try:
    df_clearing_house = spark.sql("""
        SELECT DISTINCT
            tra.ClaimHeaderId,
            NULLIF(REGEXP_REPLACE(batlot.BatchLotName, '[^a-zA-Z0-9 ]', ''), '') AS BatchLotName,
            batlot.FromDate AS BatchLotFromDate,
            tralot.ReceivedDate AS TransactionLotReceivedDate,
            tralot.Number AS TransactionLotNumber,
            tra.InvoiceNumber AS TransactionInvoiceNumber,
            NULLIF(REGEXP_REPLACE(batlot.BatchLotName, '[^a-zA-Z0-9 ]', ''), '') AS LotNumber,
            CASE WHEN tra.ClaimHeaderId IS NULL THEN 0 ELSE 1 END AS IsAutomatic
        FROM Bronze.Clearinghouse_Invoice_Transaction tra
        INNER JOIN vw_TClaimDetail CD ON tra.ClaimHeaderId = CD.HeaderId
        LEFT JOIN Bronze.Clearinghouse_Invoice_TransactionLine traline ON traline.TransactionGUI = tra.TransactionGUI
        LEFT JOIN Bronze.Clearinghouse_Invoice_TransactionLot tralot ON tralot.TransactionLotGUI = tra.TransactionLotGUI
        LEFT JOIN Bronze.Clearinghouse_Invoice_BatchLot batlot ON batlot.BatchLotGUI = tralot.BatchLotGUI
        WHERE traline.ClaimLineDetailId IS NOT NULL
    """)
    df_clearing_house.createOrReplaceTempView("vw_ClearingHouse")
    print(f"[OK] vw_ClearingHouse created — {df_clearing_house.count():,} rows")
except Exception as e:
    spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS ClaimHeaderId, CAST(NULL AS STRING) AS BatchLotName,
               CAST(NULL AS DATE) AS BatchLotFromDate, CAST(NULL AS DATE) AS TransactionLotReceivedDate,
               CAST(NULL AS STRING) AS TransactionLotNumber, CAST(NULL AS STRING) AS TransactionInvoiceNumber,
               CAST(NULL AS STRING) AS LotNumber, 0 AS IsAutomatic WHERE 1 = 0
    """).createOrReplaceTempView("vw_ClearingHouse")
    print(f"[WARN] Clearinghouse tables not available — using empty stub: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 27, Finished, Available, Finished, False)

[OK] vw_ClearingHouse created — 288,263 rows


In [26]:
# ============================================================
# Vista temporal: SubLineDetailValue (WithHolding + VAT por línea)
# Equivale a #tmpClaimSubLineDetailValue del SP
# ============================================================
df_subline_detail = spark.sql("""
    SELECT cld.ClaimLineDetailId, cld.ClaimDetailId, cd.HeaderId,
        SUM(CASE WHEN cld.ServiceIndicatorId = 230 THEN cld.ProviderLiabilityAmount + (CASE WHEN sl.InelCodeId = 15114 THEN sl.IneligibleAmount ELSE 0 END) ELSE 0 END) AS BaseWithHolding,
        SUM(CASE WHEN cld.ServiceIndicatorId = 203 THEN cld.NetCoveredAmount ELSE 0 END) AS BaseVAT
    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
    INNER JOIN vw_TClaimDetail TCD ON cd.HeaderId = TCD.HeaderId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Log cl ON cl.ClaimDetailId = cd.ClaimDetailId AND cl.ToDate IS NULL
    LEFT JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cld.ClaimDetailId = cd.ClaimDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_SubLineDetail sl ON cld.ClaimLineDetailId = sl.ClaimLineDetailId
    LEFT JOIN Bronze.AmigosPlus_AMP_Common_IneligibleCode ic ON sl.InelCodeId = ic.InelCodeId
        AND ic.InelCodeId IN (139, 143, 145, 148, 165, 308, 315, 368, 399, 505, 568, 569, 579, 601, 604, 605, 15004, 15062, 15099, 159, 15113, 15114)
    WHERE cd.HeaderId NOT IN (1107843, 2140356)
        AND cd.ServiceIndicatorId NOT IN (432)
        AND cl.ReasonID NOT IN (1094, 1555, 1577, 1721, 1858, 1859)
    GROUP BY cld.ClaimLineDetailId, cld.ClaimDetailId, cd.HeaderId
""")
df_subline_detail.createOrReplaceTempView("vw_SubLineDetailValue")
print(f"[OK] vw_SubLineDetailValue created — {df_subline_detail.count():,} rows")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 28, Finished, Available, Finished, False)

[OK] vw_SubLineDetailValue created — 3,977,114 rows


In [27]:
# ============================================================
# Vista temporal: DetailOverride (deductible override info)
# Equivale a #tempOve + #tempCode + JOIN a AMP_Claim_DetailOverride del SP
# ============================================================
df_detail_override = spark.sql("""
    SELECT CDO.ClaimDetailOverrideId, CDO.ClaimDetailId, CDO.CreatedBy, CDO.CreatedOn,
           CDO.ApprovedBy, CDO.ApprovedOn,
           ROW_NUMBER() OVER (PARTITION BY CDO.ClaimDetailId ORDER BY CDO.ClaimDetailOverrideId DESC) AS CONT
    FROM Bronze.AmigosPlus_AMP_Claim_DetailOverride CDO
""")
df_detail_override.createOrReplaceTempView("vw_DetailOverride")

# Min CreatedOn por ClaimDetailId que tenga DedNoteId
df_override_min = spark.sql("""
    SELECT cdo.ClaimDetailId,
           MIN(cdo.CreatedOn) AS CreatedOn,
           MAX(cdo.ClaimDetailOverrideId) AS ClaimDetailOverrideId
    FROM Bronze.AmigosPlus_AMP_Claim_DetailOverride cdo
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_LineDetailOverride ldo ON ldo.ClaimDetailOverrideId = cdo.ClaimDetailOverrideId
    WHERE ldo.DedNoteId IS NOT NULL
    GROUP BY cdo.ClaimDetailId
""")
df_override_min.createOrReplaceTempView("vw_OverrideMin")

# Reason code del override
df_override_reason = spark.sql("""
    SELECT a.ClaimDetailOverrideId, a.DedReasonId,
           ROW_NUMBER() OVER (PARTITION BY a.ClaimDetailOverrideId ORDER BY a.DedReasonId DESC) AS CONT
    FROM Bronze.AmigosPlus_AMP_Claim_LineDetailOverride a
""")
df_override_reason.createOrReplaceTempView("vw_OverrideReason")

print("[OK] vw_DetailOverride, vw_OverrideMin, vw_OverrideReason created")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 29, Finished, Available, Finished, False)

[OK] vw_DetailOverride, vw_OverrideMin, vw_OverrideReason created


In [28]:
# ============================================================
# Vista temporal: HighCostClaim (claims de alto costo por medicación)
# Equivale a #claimHighCost del SP — PDOCM en TAT o ValidationLogStep=5
# ============================================================

# ── Subquery 1: TAT con Reason PDOCM (solo si Gold.bgla_FactClaimTAT existe) ──
sql_parts = []

if spark.catalog.tableExists("Gold.bgla_FactClaimTAT"):
    sql_parts.append("""
        SELECT DISTINCT TAT.ClaimHeaderId
        FROM Gold.bgla_FactClaimTAT TAT
        INNER JOIN vw_TClaimDetail CD ON TAT.ClaimHeaderId = CD.HeaderId
        WHERE TAT.Iterations <> 0 AND TAT.Reason = 'PDOCM'
    """)
    print("[OK] Gold.bgla_FactClaimTAT found — included in HighCostClaim")
else:
    print("[WARN] Gold.bgla_FactClaimTAT not found — skipped from HighCostClaim")

# ── Subquery 2: ValidationLogStepId = 5 ──
try:
    sql_parts.append("""
        SELECT DISTINCT cd.HeaderId AS ClaimHeaderId
        FROM vw_TClaimDetail CDl
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Detail cd ON CDl.HeaderId = cd.HeaderId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_ValidationLogClaim clv ON clv.ClaimDetailId = cd.ClaimDetailId
        WHERE clv.ValidationLogStepId = 5
    """)

    if sql_parts:
        full_sql = " UNION ".join(sql_parts)
        df_high_cost = spark.sql(f"SELECT DISTINCT ClaimHeaderId FROM ({full_sql}) T")
    else:
        df_high_cost = spark.sql("SELECT CAST(-1 AS BIGINT) AS ClaimHeaderId WHERE 1 = 0")

    df_high_cost.createOrReplaceTempView("vw_HighCostClaim")
    print(f"[OK] vw_HighCostClaim created — {df_high_cost.count():,} rows")
except Exception as e:
    spark.sql("SELECT CAST(-1 AS BIGINT) AS ClaimHeaderId WHERE 1 = 0").createOrReplaceTempView("vw_HighCostClaim")
    print(f"[WARN] HighCostClaim dependencies not available — using empty stub: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 30, Finished, Available, Finished, False)

[WARN] Gold.bgla_FactClaimTAT not found — skipped from HighCostClaim
[OK] vw_HighCostClaim created — 26,836 rows


In [29]:
# ============================================================
# Vista temporal: Coverage (CoverageId desde BISI)
# Equivale a JOIN con BISI.CoveragesByBenefitsByPlans del SP
# ============================================================
try:
    df_coverage = spark.sql("""
        SELECT linedetail.ClaimDetailId, linedetail.ClaimLineDetailId,
               COALESCE(cover.CoverageId, -101) AS CoverageId
        FROM Bronze.AmigosPlus_AMP_Claim_LineDetail linedetail
        LEFT JOIN Bronze.BISI_CoveragesByBenefitsByPlans cbp ON cbp.CoveragesByBenefitsByPlansKey = linedetail.CoveragebyBenefitbyPlanKey
        LEFT JOIN Bronze.BISI_CoveragesByBenefits cb ON cbp.CoverageByBenefitsKey = cb.CoverageByBenefitsKey
        LEFT JOIN Bronze.BISI_Coverage cover ON cb.CoverageKey = cover.CoverageKey
    """)
    df_coverage.createOrReplaceTempView("vw_Coverage")
    print(f"[OK] vw_Coverage created — {df_coverage.count():,} rows")
except Exception as e:
    spark.sql("""
        SELECT CAST(-1 AS INT) AS ClaimDetailId, CAST(-1 AS BIGINT) AS ClaimLineDetailId,
               CAST(-101 AS INT) AS CoverageId WHERE 1 = 0
    """).createOrReplaceTempView("vw_Coverage")
    print(f"[WARN] BISI tables not available — using empty stub: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 31, Finished, Available, Finished, False)

[OK] vw_Coverage created — 12,944,329 rows


In [30]:
# ============================================================
# Vista temporal: PostedDate (fecha de PostedOn desde JournalEntry)
# Equivale a #Posteddate del SP usp_DataPrepare_FactClaim
# Cadena: ClaimDetail → DetailRun → Run → PaymentLog → JournalEntry
# ============================================================
try:
    df_posted_date = spark.sql("""
        SELECT cd.HeaderId, cd.ClaimDetailId,
               MAX(je.PostedOn) AS PostedOn
        FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
        INNER JOIN vw_TClaimDetail tCD ON tCD.HeaderId = cd.HeaderId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_DetailRun cdr ON cd.ClaimDetailId = cdr.ClaimDetailId
        INNER JOIN Bronze.AmigosPlus_AMP_Claim_Run run ON cdr.RunId = run.RunId
        LEFT JOIN Bronze.AmigosPlus_AMP_Common_PaymentLog paymentlog ON paymentlog.PaymentId = run.PaymentId
        LEFT JOIN Bronze.AmigosPlus_AMP_Acct_JournalEntry je ON paymentlog.JournalEntryId = je.JournalEntryId
        GROUP BY cd.HeaderId, cd.ClaimDetailId
    """)
    df_posted_date.createOrReplaceTempView("vw_PostedDate")
    print(f"[OK] vw_PostedDate created — {df_posted_date.count():,} rows")
except Exception as e:
    spark.sql("""
        SELECT CAST(-1 AS BIGINT) AS HeaderId, CAST(-1 AS INT) AS ClaimDetailId,
               CAST(NULL AS TIMESTAMP) AS PostedOn WHERE 1 = 0
    """).createOrReplaceTempView("vw_PostedDate")
    print(f"[WARN] PostedDate dependencies not available — using empty stub: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 32, Finished, Available, Finished, False)

[OK] vw_PostedDate created — 6,604,540 rows


In [31]:
# ============================================================
# PolicyHistory: dimensiones de póliza desde Silver.tmp_policyamigosplushistory
# Fuente migrada de __TMPvwTMPBGLAEDW___PolicyAmigosPlusHistory
# ============================================================
df_policy_history = spark.sql("""
    SELECT DISTINCT
        pp.PolicySKey_PolicyId AS PolicyId,
        COALESCE(pp.ModeOfPaymentSKey_ModeOfPaymentId, -101) AS ModeOfPaymentId,
        COALESCE(pp.PaymentMethodSKey_PaymentMethodId, -1) AS PaymentMethodId,
        COALESCE(pp.ReceivedMethodSKey_ReceivedMethodId, -101) AS PolicyReceivedMethodId,
        COALESCE(pp.GroupSKey_GroupId, -101) AS GroupId,
        COALESCE(pp.FamilyTypeSKey_FamilyTypeId, -101) AS FamilyTypeId,
        COALESCE(pp.ProductSKey_ProductId, -101) AS ProductId,
        COALESCE(pp.EntitySKey_EntityId, -101) AS EntityId,
        COALESCE(pp.PlanSKey_PlanId, -101) AS PlanId,
        COALESCE(pp.BusinessModeSKey_BusinessModeId, -101) AS BusinessModeId,
        COALESCE(pp.BusinessTypeSKey_BusinessTypeId, -101) AS BusinessTypeId,
        COALESCE(pp.InsuranceBusinessSKey_InsuranceBusinessId, -101) AS InsuranceBusinessId,
        COALESCE(pp.CompanySKey_CompanyId, -101) AS CompanyId,
        pp.PolicyIssueDateKey_Date AS PolicyIssueDateRaw,
        pp.RenewalDateKey_Date AS PolicyRenewalDateRaw,
        pp.AnniversaryDateKey_Date AS PolicyAnniversaryDateRaw,
        pp.ApplicationReceivedDateKey_Date AS PolicyApplicationReceivedDateRaw,
        pp.PolicyEffectiveDateKey_Date AS PolicyEffectiveDateRaw,
        pp.PolicyCancelDateKey_Date AS PolicyCancelDateRaw,
        pp.DeathBenefitperiodDateKey_Date AS PolicyDeathBenefitPeriodDateRaw,
        COALESCE(pp.AgentSKey_AgentHierarchyId, -101) AS AgentHierarchyId,
        COALESCE(pp.RegionSkey_RegionId, -101) AS RegionId
    FROM Silver.tmp_policyamigosplushistory pp
""")
df_policy_history.createOrReplaceTempView("vw_PolicyHistory")
print(f"[OK] vw_PolicyHistory created from Silver.tmp_policyamigosplushistory — {df_policy_history.count():,} rows")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 33, Finished, Available, Finished, False)

[OK] vw_PolicyHistory created from Silver.tmp_policyamigosplushistory — 786,437 rows


In [ ]:
# ============================================================
# Staging: INSERT INTO Silver.TmpFactClaim
# Equivale al INSERT principal del SP usp_DataPrepare_FactClaim
# Todos los campos monetarios/numéricos se proyectan como DOUBLE
# ============================================================

df_staging = spark.sql("""
    SELECT
        -- ── Keys ──
        cd.ClaimDetailId,
        TCD.LogId,
        cld.ClaimLineDetailId,
        cld.SeqNo                                                                                    AS ClaimLineSequenceNumber,
        DENSE_RANK() OVER (PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC)                 AS ClaimVersion,
        ch.ReferralHeaderId,
        ch.ClaimUUID,
        cd.HeaderId,
        ch.Number,

        -- ── Dimension source IDs ──
        cl.StatusId,
        cl.ReasonId,
        CAST(cl.FromDate AS DATE)                                                                    AS StatusReasonDate,
        COALESCE(me.MemberEligibilityId, -101)                                                       AS MemberEligibilityId,
        cd.BillingProviderId,
        cld.ServiceProviderId,
        cld.ScheduleId,
        CAST(svc.ServiceFromDate AS DATE)                                                            AS ServiceFromDate,
        cld.ToDate                                                                                   AS ServiceToDate,
        cd.ServiceIndicatorId                                                                        AS DetailServiceIndicator,
        cld.ServiceIndicatorId                                                                       AS LineDetailServiceIndicator,
        cld.ReportingServiceIndicatorId                                                              AS ReportingServIndicatorId,
        diag.DiagnosticId,
        proc.ProcedureId,
        cld.PlaceOfServiceId,
        cld.TypeOfServiceId,
        cld.PriorityId,
        COALESCE(cld.GlobalLocationId, -101)                                                         AS GlobalLocationId,
        COALESCE(cld.CountryId, -101)                                                               AS CountryId,
        COALESCE(cld.StateId, -101)                                                                 AS StateId,
        COALESCE(cld.CityId, -101)                                                                  AS CityId,
        COALESCE(cld.AccumulatorScopeId, -101)                                                      AS AccumulatorScopeId,
        COALESCE(ch.BillTypeId, -101)                                                               AS BillTypeId,
        cd.PayTo,
        COALESCE(py.PayeeId, -101)                                                                  AS PayeeId,

        -- ── Claim fields ──
        ch.AccountNo,
        CAST(cd.MemberPPA AS DOUBLE)                                                                AS MemberPPA,
        CAST(cd.ProviderPPA AS DOUBLE)                                                              AS ProviderPPA,
        ch.RefNo,
        COALESCE(cd.Penalty, 0)                                                                     AS Penalty,
        CAST(cld.Quantity AS DOUBLE)                                                                AS Quantity,

        -- ── Currency source IDs ──
        COALESCE(cd.LocalCurrencyId, -101)                                                          AS LocalCurrencyId,
        COALESCE(pbc.BaseCurrencyId, -101)                                                          AS BaseCurrencyId,
        COALESCE(cld.MemberPaymentCurrencyId, -101)                                                 AS MemberPaymentCurrencyId,
        COALESCE(cld.ProviderPaymentCurrencyId, -101)                                               AS ProviderPaymentCurrencyId,
        CAST(cld.XchangeRateDate AS DATE)                                                           AS XchangeRateDate,
        CAST(cld.ExchangeRate AS DOUBLE)                                                            AS ClaimXchangeRate,

        -- ── Amounts (DOUBLE) ──
        CAST(cld.BilledAmount AS DOUBLE)                                                            AS ClaimBilledAmountLocalCurrency,
        CAST(cld.BilledAmountUSD AS DOUBLE)                                                         AS ClaimBilledAmountLocalCurrencyUSD,
        CAST(cld.BilledAmountBaseCurrency AS DOUBLE)                                                AS ClaimBilledAmountBaseCurrency,
        CAST(cld.BilledAmountBaseCurrencyUSD AS DOUBLE)                                             AS ClaimBilledAmountBaseCurrencyUSD,
        CAST(cld.IneligibleAmount AS DOUBLE)                                                        AS ClaimIneligAmount,
        CAST(cld.IneligibleAmountUSD AS DOUBLE)                                                     AS ClaimIneligAmountUSD,
        CAST(cld.PrepaidAmount AS DOUBLE)                                                           AS ClaimPrepaidAmount,
        CAST(cld.PrepaidAmountUSD AS DOUBLE)                                                        AS ClaimPrepaidAmountUSD,
        CAST(cld.CoInsuranceAmount AS DOUBLE)                                                       AS ClaimCoInsuranceAmount,
        CAST(cld.CoInsuranceAmountUSD AS DOUBLE)                                                    AS ClaimCoInsuranceAmountUSD,
        CAST(cld.DeductibleAmount AS DOUBLE)                                                        AS ClaimDeductibleAmount,
        CAST(cld.DeductibleAmountUSD AS DOUBLE)                                                     AS ClaimDeductibleAmountUSD,
        CAST(cld.AllowedAmount AS DOUBLE)                                                           AS ClaimAllowedAmount,
        CAST(cld.AllowedAmountUSD AS DOUBLE)                                                        AS ClaimAllowedAmountUSD,
        CAST(cld.CoveredAmount AS DOUBLE)                                                           AS ClaimCoveredAmount,
        CAST(cld.CoveredAmountUSD AS DOUBLE)                                                        AS ClaimCoveredAmountUSD,
        CAST(cld.NetCoveredAmount AS DOUBLE)                                                        AS ClaimNetCoveredAmount,
        CAST(cld.NetCoveredAmountUSD AS DOUBLE)                                                     AS ClaimNetCoveredAmountUSD,
        CAST(cld.DiscountsAmount AS DOUBLE)                                                         AS ClaimDiscountsAmount,
        CAST(cld.DiscountsAmountUSD AS DOUBLE)                                                      AS ClaimDiscountsAmountUSD,
        CAST(cld.NonCoveredAmount AS DOUBLE)                                                        AS ClaimNonCoveredAmount,
        CAST(cld.NonCoveredAmountUSD AS DOUBLE)                                                     AS ClaimNonCoveredAmountUSD,
        CAST(cld.MemberLiabilityAmount AS DOUBLE)                                                   AS ClaimMemberLiabilityAmount,
        CAST(cld.MemberLiabilityAmountUSD AS DOUBLE)                                                AS ClaimMemberLiabilityAmountUSD,
        CAST(cld.ProviderLiabilityAmount AS DOUBLE)                                                 AS ClaimProviderLiabilityAmount,
        CAST(cld.ProviderLiabilityAmountUSD AS DOUBLE)                                              AS ClaimProviderLiabilityAmountUSD,
        CAST(cld.MemberPaymentAmount AS DOUBLE)                                                     AS ClaimMemberPaymentAmount,
        CAST(cld.MemberPaymentAmountUSD AS DOUBLE)                                                  AS ClaimMemberPaymentAmountUSD,
        CAST(cld.MemberPaymentAmountCurrency AS DOUBLE)                                             AS ClaimMemberPaymentAmountCurrency,
        CAST(cld.MemberPaymentAmountCurrencyUSD AS DOUBLE)                                          AS ClaimMemberPaymentAmountCurrencyUSD,
        CAST(cld.MemberPaymentExchangeRate AS DOUBLE)                                               AS ClaimMemberPaymentXchangeRate,
        CAST(cld.ProviderPaymentAmount AS DOUBLE)                                                   AS ClaimProviderPaymentAmount,
        CAST(cld.ProviderPaymentAmountUSD AS DOUBLE)                                                AS ClaimProviderPaymentAmountUSD,
        CAST(cld.ProviderPaymentAmountCurrency AS DOUBLE)                                           AS ClaimProviderPaymentAmountCurrency,
        CAST(cld.ProviderPaymentAmountCurrencyUSD AS DOUBLE)                                        AS ClaimProviderPaymentAmountCurrencyUSD,
        CAST(cld.ProviderPaymentExchangeRate AS DOUBLE)                                             AS ClaimProviderPaymentXchangeRate,
        CAST(cld.COBPrePaidAmount AS DOUBLE)                                                        AS ClaimCOBPrePaidAmount,
        CAST(cld.COBPrePaidAmountUSD AS DOUBLE)                                                     AS ClaimCOBPrePaidAmountUSD,
        CAST(cld.OtherPrePaidAmount AS DOUBLE)                                                      AS ClaimOtherPrePaidAmount,
        CAST(cld.OtherPrePaidAmountUSD AS DOUBLE)                                                   AS ClaimOtherPrePaidAmountUSD,

        -- ── Processed ──
        CAST(pd.FromDate AS DATE)                                                                    AS ProcessedDate,
        CAST(DENSE_RANK() OVER (PARTITION BY cd.HeaderId ORDER BY cd.ClaimDetailId DESC) AS STRING) AS ControlVersion,
        COALESCE(pd.StatusId, -101)                                                                 AS ClaimProcessedStatusId,
        COALESCE(pd.ReasonId, -101)                                                                 AS ClaimProcessedReasonId,

        -- ── Submission ──
        cld.InOnNetwork,
        COALESCE(cs.InboundChannel, '_Undefined')                                                   AS SubmissionInboundChannel,
        COALESCE(cs.TrackingNumber, '_Undefined')                                                   AS SubmissionTrackingNumber,
        COALESCE(CAST(cs.SenderTypeId AS STRING), '_Undefined')                                     AS SubmissionSenderType,
        COALESCE(cs.SenderEmailAddress, '_Undefined')                                               AS SubmissionSenderEmailAddress,
        COALESCE(cs.SenderFullName, '_Undefined')                                                   AS SubmissionSenderFullName,
        CAST(cs.ReceivedDate AS DATE)                                                               AS SubmissionReceivedDate,
        COALESCE(cs.IsComplement, FALSE)                                                            AS SubmissionIsComplement,
        COALESCE(cld.ServiceNetwork, '_Undefined')                                                  AS ServiceNetwork,

        -- ── Business Flags ──
        0                                                                                           AS ClaimGap,
        COALESCE(rmd.ReceivedMethodId, -101)                                                        AS ClaimReceivedMethodId,
        COALESCE(cd.PolicyFirstYear, 0)                                                             AS PolicyFirstYear,
        COALESCE(cd.PolicyFirstMonth, 0)                                                            AS PolicyFirstMonth,
        0                                                                                           AS FirstClaim,
        CASE WHEN rmd.ReceivedMethodId IS NOT NULL THEN 1 ELSE 0 END                                AS IsDigital,
        COALESCE(cld.IsFastTrack, FALSE)                                                            AS IsFastTrack,
        COALESCE(CAST(cld.HTHFeePaid AS DOUBLE), 0)                                                AS HTHFeePaid,
        COALESCE(CAST(cld.HTHFeePercent AS DOUBLE), 0)                                              AS HTHFeePercent,
        0                                                                                           AS IsDeleted,
        COALESCE(cld.ProviderNetworkClassName, '_Undefined')                                        AS ProviderNetworkClassName,
        COALESCE(CAST(cld.BrackedClaimKey AS STRING), '0')                                         AS BrackedClaimKey,
        COALESCE(CAST(cld.BrackedMemberKey AS STRING), '0')                                        AS BrackedMemberKey,
        CAST(cld.IncomebyinsuredIA AS DOUBLE)                                                       AS IncomebyinsuredIA,
        CAST(cld.SpendbyinsuredGA AS DOUBLE)                                                        AS SpendbyinsuredGA,
        COALESCE(cld.ClaimValidation, '_Undefined')                                                 AS ClaimValidation,

        -- ── Clinical Flags ──
        COALESCE(cld.IsICU, 0)                                                                     AS IsICU,
        COALESCE(cld.IsEmergency, 0)                                                               AS IsEmergency,
        0                                                                                           AS IsHospital_Emergency,
        COALESCE(cld.IsChildbirths, 0)                                                             AS IsChildbirths,
        COALESCE(cld.IsCesareanSection, 0)                                                         AS IsCesareanSection,

        -- ── ClearingHouse / Lot ──
        COALESCE(ch_lot.BatchLotName, '_Undefined')                                                 AS BatchLotName,
        ch_lot.BatchLotFromDate,
        ch_lot.TransactionLotReceivedDate,
        COALESCE(ch_lot.TransactionLotNumber, '_Undefined')                                         AS TransactionLotNumber,
        COALESCE(ch_lot.TransactionInvoiceNumber, '_Undefined')                                     AS TransactionInvoiceNumber,
        COALESCE(ch_lot.LotNumber, '_Undefined')                                                    AS LotNumber,
        COALESCE(ch_lot.IsAutomatic, 0)                                                             AS IsAutomatic,
        COALESCE(cd.FirstYearClaimExceptions, '_Undefined')                                         AS FirstYearClaimExceptions,
        0                                                                                           AS ClaimsOutliers,

        -- ── Source IDs ──
        cd.PolicyId,
        me.MemberId,

        -- ── WithHolding / VAT ──
        COALESCE(CAST(csl.BaseWithHolding AS DOUBLE), 0)                                            AS BaseWithHolding,
        COALESCE(CAST(csl.BaseVAT AS DOUBLE), 0)                                                    AS BaseVAT,
        COALESCE(cd.ThirdPartyId, -101)                                                             AS ThirdPartyId,

        -- ── TAT ──
        COALESCE(tat.LocalTeam, '_Non Defined LocalTeam')                                           AS LocalTeam,
        COALESCE(tat.IsCOR, '_Not defined IsCOR')                                                   AS IsCOR,

        -- ── BISI Coinsurance ──
        CAST(COALESCE(coin.CoinsurancePercent, '0') AS DOUBLE)
            * CAST(cld.AllowedAmount AS DOUBLE) / 100.0                                             AS ClaimCoInsuranceBISIAmount,
        CAST(COALESCE(coin.CoinsurancePercent, '0') AS DOUBLE)
            * CAST(cld.AllowedAmountUSD AS DOUBLE) / 100.0                                         AS ClaimCoInsuranceBISIAmountUSD,
        COALESCE(cov.CoverageId, -101)                                                              AS CoverageId,
        CASE WHEN hcc.ClaimHeaderId IS NOT NULL THEN 1 ELSE 0 END                                   AS HigthCostClaim,

        -- ── Payment Posted ──
        CAST(ppd.PostedOn AS DATE)                                                                  AS PaymentPostedDate,

        -- ── Deductible Override ──
        CAST(ovmin.CreatedOn AS DATE)                                                               AS DeductibleOverrideCreatedOn,
        COALESCE(ov.CreatedBy, '_Undefined')                                                        AS DeductibleOverrideCreatedBy,
        COALESCE(cd.DeductibleOverrideCreatedByUserFullName, '_Undefined')                          AS DeductibleOverrideCreatedByUserFullName,
        CAST(ov.ApprovedOn AS DATE)                                                                 AS DeductibleOverrideApprovedOn,
        COALESCE(ov.ApprovedBy, '_Undefined')                                                       AS DeductibleOverrideApprovedBy,
        COALESCE(cd.DeductibleOverrideApprovedByUserName, '_Undefined')                             AS DeductibleOverrideApprovedByUserName,
        COALESCE(CAST(orcode.DedReasonId AS STRING), '_Undefined')                                  AS DeductibleOverrideReasonCode,
        COALESCE(cd.DeductibleOverrideReasonDescription, '_Undefined')                              AS DeductibleOverrideReasonDescription,

        -- ── Misc ──
        COALESCE(cl.UpdatedBy, '_Undefined')                                                        AS UpdatedBy,
        COALESCE(ch.ClaimType, '_Undefined')                                                        AS ClaimType,
        CAST(ch.IssuedDate AS DATE)                                                                 AS IssuedDate,
        CAST(cl.FromDate AS DATE)                                                                   AS ActionDate,

        -- ── Policy Dimension source IDs (from vw_PolicyHistory) ──
        COALESCE(ph.EntityId, -101)                                                                 AS PolicyEntityId,
        COALESCE(ph.GroupId, -101)                                                                  AS PolicyGroupId,
        COALESCE(ph.ProductId, -101)                                                                AS PolicyProductId,
        COALESCE(ph.PlanId, -101)                                                                   AS PolicyPlanId,
        COALESCE(ph.BusinessModeId, -101)                                                           AS PolicyBusinessModeId,
        COALESCE(ph.BusinessTypeId, -101)                                                           AS PolicyBusinessTypeId,
        COALESCE(ph.InsuranceBusinessId, -101)                                                      AS PolicyInsuranceBusinessId,
        COALESCE(ph.CompanyId, -101)                                                                AS PolicyCompanyId,
        COALESCE(ph.AgentHierarchyId, -101)                                                         AS PolicyAgentHierarchyId,
        COALESCE(ph.RegionId, -101)                                                                 AS PolicyRegionId,
        COALESCE(ph.ModeOfPaymentId, -101)                                                          AS PolicyModeOfPaymentId,
        COALESCE(ph.PaymentMethodId, -101)                                                          AS PolicyPaymentMethodId,
        COALESCE(ph.PolicyReceivedMethodId, -101)                                                   AS PolicyReceivedMethodId,
        COALESCE(ph.FamilyTypeId, -101)                                                             AS PolicyFamilyTypeId,

        -- ── Policy Date Keys (raw dates for DimDate lookup) ──
        CAST(ph.PolicyIssueDateRaw AS DATE)                                                         AS PolicyIssueDateRaw,
        CAST(ph.PolicyRenewalDateRaw AS DATE)                                                       AS PolicyRenewalDateRaw,
        CAST(ph.PolicyAnniversaryDateRaw AS DATE)                                                   AS PolicyAnniversaryDateRaw,
        CAST(ph.PolicyApplicationReceivedDateRaw AS DATE)                                           AS PolicyApplicationReceivedDateRaw,
        CAST(ph.PolicyEffectiveDateRaw AS DATE)                                                     AS PolicyEffectiveDateRaw,
        CAST(ph.PolicyCancelDateRaw AS DATE)                                                        AS PolicyCancelDateRaw,
        CAST(ph.PolicyDeathBenefitPeriodDateRaw AS DATE)                                            AS PolicyDeathBenefitPeriodDateRaw,

        -- ── VAT / Withholding in currency ──
        CAST(csl.BaseVAT AS DOUBLE)                                                                 AS VATBaseCurrenty,
        CAST(csl.BaseVAT AS DOUBLE)                                                                 AS VATUSDCurrenty,
        CAST(csl.BaseWithHolding AS DOUBLE)                                                         AS WithholdingBaseCurrenty,
        CAST(csl.BaseWithHolding AS DOUBLE)                                                         AS WithholdingUSDCurrenty,

        -- ── ClaimUpdatedByUser + Source ──
        COALESCE(cl.UpdatedBy, '_Undefined')                                                        AS ClaimUpdatedByUser,
        'AmigosPlus'                                                                                AS Source,

        -- ── Date fields for dim resolution ──
        CAST(rcv.ReceivedDate AS DATE)                                                              AS ReceivedDate,

        -- ── SCD Fields ──
        1                                                                                           AS ActualRowFlag,
        current_date()                                                                              AS EffectiveDate,
        current_timestamp()                                                                         AS LoadDate

    FROM Bronze.AmigosPlus_AMP_Claim_Detail cd
    INNER JOIN vw_TClaimDetail TCD                  ON TCD.HeaderId              = cd.HeaderId
    INNER JOIN Bronze.AmigosPlus_AMPview_Claim_LineDetail cld ON cld.ClaimDetailId = cd.ClaimDetailId
    INNER JOIN Bronze.AmigosPlus_AMP_Claim_Header ch ON ch.HeaderId              = cd.HeaderId
    INNER JOIN vw_ClaimLog cl                       ON cl.ClaimDetailId          = cd.ClaimDetailId AND cl.rn = 1
    INNER JOIN vw_CurrentVersion cv                 ON cv.ClaimDetailId          = cd.ClaimDetailId
    LEFT  JOIN vw_TmpServices svc                   ON svc.HeaderId              = cd.HeaderId
    LEFT  JOIN vw_TmpReceived rcv                   ON rcv.HeaderId              = cd.HeaderId AND rcv.Cont = 1
    LEFT  JOIN vw_DiagnosticLine diag               ON diag.ClaimLineDetailId    = cld.ClaimLineDetailId AND diag.ID = 1
    LEFT  JOIN vw_ProcedureLine proc                ON proc.ClaimLineDetailId    = cld.ClaimLineDetailId AND proc.ID = 1
    LEFT  JOIN Bronze.AmigosPlus_AMP_Policy_MemberEligibility me ON me.MemberEligibilityId = cd.MemberEligibilityId
    LEFT  JOIN vw_PolicyBaseCurrency pbc            ON pbc.MemberEligibilityId   = cd.MemberEligibilityId
    LEFT  JOIN vw_Payee py                          ON py.ContactBaseId          = cd.PayeeContactBaseId AND py.RowNum = 1
    LEFT  JOIN vw_ClaimSubmission cs                ON cs.ClaimHeaderId          = cd.HeaderId AND cs.CONT = 1
    LEFT  JOIN vw_ReceivedMethodDigital rmd         ON rmd.ReceivedMethodId      = cd.ReceivedMethodId
    LEFT  JOIN vw_SubLineDetailValue csl            ON csl.ClaimLineDetailId     = cld.ClaimLineDetailId
                                                    AND csl.ClaimDetailId        = cd.ClaimDetailId
    LEFT  JOIN vw_ProcessedDate pd                  ON pd.ClaimDetailId          = cd.ClaimDetailId AND pd.Cont = 1
    LEFT  JOIN vw_LocalTeamTAT tat                  ON tat.ClaimHeaderId         = cd.HeaderId
    LEFT  JOIN vw_HighCostClaim hcc                 ON hcc.ClaimHeaderId         = cd.HeaderId
    LEFT  JOIN vw_CoinsurancePercent coin           ON coin.ClaimDetailId        = cd.ClaimDetailId
                                                    AND coin.ClaimLineDetailId   = cld.ClaimLineDetailId
    LEFT  JOIN vw_ClearingHouse ch_lot              ON ch_lot.ClaimHeaderId      = cd.HeaderId
    LEFT  JOIN vw_Coverage cov                      ON cov.ClaimDetailId         = cd.ClaimDetailId
                                                    AND cov.ClaimLineDetailId    = cld.ClaimLineDetailId
    LEFT  JOIN vw_PostedDate ppd                    ON ppd.HeaderId              = cd.HeaderId
                                                    AND ppd.ClaimDetailId        = cd.ClaimDetailId
    LEFT  JOIN vw_DetailOverride ov                 ON ov.ClaimDetailId          = cd.ClaimDetailId AND ov.CONT = 1
    LEFT  JOIN vw_OverrideMin ovmin                 ON ovmin.ClaimDetailId       = cd.ClaimDetailId
    LEFT  JOIN vw_OverrideReason orcode             ON orcode.ClaimDetailOverrideId = ov.ClaimDetailOverrideId AND orcode.CONT = 1
    LEFT  JOIN vw_PolicyHistory ph                  ON ph.PolicyId               = cd.PolicyId
""")

df_staging.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("Silver.TmpFactClaim")
print(f"[OK] Silver.TmpFactClaim populated — {df_staging.count():,} rows")


StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 34, Finished, Available, Finished, False)

[OK] Silver.TmpFactClaim loaded — 12,945,318 rows


In [33]:
# ============================================================
# Pre-cache dimension tables for broadcast joins
# ============================================================
from pyspark.sql.functions import broadcast

dim_tables = [
    ("Gold.bgla_DimStatus",              "vw_DimStatus_bc"),
    ("Gold.bgla_DimClaimReason",          "vw_DimClaimReason_bc"),
    ("Gold.bgla_DimDate",                 "vw_DimDate_bc"),
    ("Gold.bgla_DimPolicyMember",         "vw_DimPolicyMember_bc"),
    ("Gold.bgla_DimProvider",             "vw_DimProvider_bc"),
    ("Gold.bgla_DimSchedule",             "vw_DimSchedule_bc"),
    ("Gold.bgla_DimServiceIndicator",     "vw_DimServiceIndicator_bc"),
    ("Gold.bgla_DimServiceNetwork",       "vw_DimServiceNetwork_bc"),
    ("Gold.bgla_DimDiagnostic",           "vw_DimDiagnostic_bc"),
    ("Gold.bgla_DimProcedure",            "vw_DimProcedure_bc"),
    ("Gold.bgla_DimPlaceOfService",       "vw_DimPlaceOfService_bc"),
    ("Gold.bgla_DimTypeOfService",        "vw_DimTypeOfService_bc"),
    ("Gold.bgla_DimPriority",             "vw_DimPriority_bc"),
    ("Gold.bgla_DimCurrency",             "vw_DimCurrency_bc"),
    ("Gold.bgla_DimGeography",            "vw_DimGeography_bc"),
    ("Gold.bgla_DimAccumulatorScope",     "vw_DimAccumulatorScope_bc"),
    ("Gold.bgla_DimBillType",             "vw_DimBillType_bc"),
    ("Gold.bgla_DimPayTo",               "vw_DimPayTo_bc"),
    ("Gold.bgla_DimPayee",               "vw_DimPayee_bc"),
    ("Gold.bgla_DimPolicy",              "vw_DimPolicy_bc"),
    ("Gold.bgla_DimReceivedMethod",       "vw_DimReceivedMethod_bc"),
    # ── NEW: Policy dimension tables ──
    ("Gold.bgla_DimEntity",               "vw_DimEntity_bc"),
    ("Gold.bgla_DimGroup",                "vw_DimGroup_bc"),
    ("Gold.bgla_DimProduct",              "vw_DimProduct_bc"),
    ("Gold.bgla_DimPlan",                 "vw_DimPlan_bc"),
    ("Gold.bgla_DimBusinessMode",         "vw_DimBusinessMode_bc"),
    ("Gold.bgla_DimBusinessType",         "vw_DimBusinessType_bc"),
    ("Gold.bgla_DimFamilyType",           "vw_DimFamilyType_bc"),
    ("Gold.bgla_DimCompany",              "vw_DimCompany_bc"),
    ("Gold.bgla_DimAgent",                "vw_DimAgent_bc"),
    ("Gold.bgla_DimRegion",               "vw_DimRegion_bc"),
    ("Gold.bgla_DimModeOfPayment",        "vw_DimModeOfPayment_bc"),
    ("Gold.bgla_DimPaymentMethod",        "vw_DimPaymentMethod_bc"),
    ("Gold.bgla_DimInsuranceBusiness",    "vw_DimInsuranceBusiness_bc"),
    ("Gold.BGLA_DimThirdParty",           "vw_DimThirdParty_bc"),
    ("Gold.BGLA_DimSeverityBracket",      "vw_DimSeverityBracket_bc"),
]

for table_name, view_name in dim_tables:
    try:
        df = spark.table(table_name)
        broadcast(df).createOrReplaceTempView(view_name)
    except Exception as e:
        print(f"  ⚠️ {table_name}: {e}")

print("[OK] Dimension tables pre-cached for broadcast joins")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 35, Finished, Available, Finished, False)

[OK] Dimension tables pre-cached for broadcast joins


In [34]:
# ============================================================
# Final view with surrogate key resolution + broadcast hints
# Incluye source IDs para InsertDataOrphan
# ============================================================
df_final = spark.sql("""
    SELECT /*+ BROADCAST(DimStatus_ClaimStatus, DimClaimReason, DimDate_StatusReason,
                          DimMember_Claim, DimProvider_Billing, DimProvider_Service,
                          DimSchedule, DimDate_ServiceFrom, DimServiceNetwork,
                          DimServiceIndicator_Line, DimServiceIndicator_Reporting,
                          DimDiagnostic, DimProcedure, DimPlaceOfService, DimTypeOfService,
                          DimPriority, DimDate_Received, DimGeography,
                          DimAccumulatorScope, DimBillType, DimPayTo, DimPayee,
                          DimPolicy, DimCurrency_Local, DimCurrency_Base,
                          DimCurrency_MemberPayment, DimCurrency_ProviderPayment,
                          DimDate_XchangeRate, DimDate_Processed,
                          DimStatus_ProcessedStatus, DimClaimReason_Processed,
                          DimDate_SubmissionReceived, DimReceivedMethod,
                          DimDate_PaymentPosted,
                          DimEntity_Policy, DimGroup_Policy, DimProduct_Policy,
                          DimPlan_Policy, DimBusinessMode_Policy, DimBusinessType_Policy,
                          DimFamilyType_Policy, DimCompany_Policy, DimAgent_Policy,
                          DimRegion_Policy, DimModeOfPayment_Policy, DimPaymentMethod_Policy,
                          DimInsuranceBusiness_Policy, DimReceivedMethod_Policy,
                          DimDate_PolicyIssue, DimDate_PolicyRenewal, DimDate_PolicyAnniversary,
                          DimDate_PolicyAppReceived, DimDate_PolicyEffective,
                          DimDate_PolicyCancel, DimDate_PolicyDeathBenefit,
                          DimThirdParty, DimSeverityBracket) */
        UUID() AS FactClaimSKey,
        TMP.ClaimDetailId,
        TMP.LogId,
        TMP.ClaimLineDetailId,
        TMP.ClaimLineSequenceNumber,
        TMP.ClaimVersion,
        TMP.ReferralHeaderId AS ClaimReferralHeaderId,
        SUBSTRING(COALESCE(TMP.ClaimUUID, '_Undefined'), 1, 100) AS ClaimUUID,
        TMP.HeaderId AS ClaimHeaderId,
        TMP.Number AS ClaimNumber,
        -- Dimension SKey lookups
        COALESCE(CAST(DimStatus_ClaimStatus.StatusSKey AS STRING), '-101') AS ClaimStatusSKey,
        COALESCE(CAST(DimClaimReason.ClaimReasonSKey AS STRING), '-101') AS ClaimReasonSKey,
        COALESCE(DimDate_StatusReason.Datekey, -101) AS ClaimStatusReasonDatekey,
        COALESCE(CAST(DimMember_Claim.PolicyMemberSKey AS STRING), '-101') AS ClaimMemberSKey,
        COALESCE(CAST(DimProvider_Billing.ProviderSKey AS STRING), '-101') AS ClaimBillingProviderSkey,
        COALESCE(CAST(DimProvider_Service.ProviderSKey AS STRING), '-101') AS ClaimServiceProviderSKey,
        COALESCE(CAST(DimSchedule.ScheduleSKey AS STRING), '-101') AS ClaimScheduleSKey,
        COALESCE(DimDate_ServiceFrom.Datekey, -101) AS ClaimServiceFromDateKey,
        TMP.ServiceToDate AS ClaimServiceToDate,
        COALESCE(CAST(DimServiceNetwork.ServiceNetworkSKey AS STRING), '-101') AS ClaimNetworkSkey,
        COALESCE(CAST(DimServiceIndicator_Line.ServiceIndicatorSKey AS STRING), '-101') AS ClaimLineDetailServiceindicatorSkey,
        COALESCE(CAST(DimServiceIndicator_Reporting.ServiceIndicatorSKey AS STRING), '-101') AS ClaimReportingServiceindicatorSkey,
        COALESCE(CAST(DimDiagnostic.DiagnosticSKey AS STRING), '-101') AS ClaimPrimaryDiagnosticSkey,
        COALESCE(CAST(DimProcedure.ProcedureSKey AS STRING), '-101') AS ClaimPrimaryProcedureSkey,
        COALESCE(CAST(DimPlaceOfService.PlaceOfServiceSKey AS STRING), '-101') AS ClaimPlaceOfServiceSKey,
        COALESCE(CAST(DimTypeOfService.TypeOfServiceSKey AS STRING), '-101') AS ClaimTypeOfServiceSKey,
        COALESCE(CAST(DimPriority.PrioritySKey AS STRING), '-101') AS ClaimPrioritySkey,
        COALESCE(DimDate_Received.Datekey, -101) AS ClaimReceivedDateKey,
        COALESCE(CAST(DimGeography.GeographySKey AS STRING), '-101') AS ClaimGeographySKey,
        COALESCE(CAST(DimAccumulatorScope.AccumulatorScopeSKey AS STRING), '-101') AS ClaimAccumulatorScopeSKey,
        COALESCE(CAST(DimBillType.BillTypeSKey AS STRING), '-101') AS ClaimBillTypeSKey,
        COALESCE(CAST(DimPayTo.PayToSKey AS STRING), '-101') AS ClaimPayToSKey,
        COALESCE(CAST(DimPayee.PayeeSkey AS STRING), '-101') AS ClaimPayeeSkey,
        TMP.AccountNo AS ClaimAccountNumber,
        COALESCE(TMP.MemberPPA, 0) AS MemberPPA,
        COALESCE(TMP.ProviderPPA, 0) AS ProviderPPA,
        COALESCE(CAST(DimPolicy.PolicySKey AS STRING), '-101') AS PolicySKey,
        TMP.RefNo AS ClaimReferenceNumber,
        COALESCE(CAST(TMP.Penalty AS INT), 0) AS ClaimPenalty,
        COALESCE(TMP.Quantity, 0) AS ClaimQuantity,
        COALESCE(CAST(DimCurrency_Local.CurrencySKey AS STRING), '-101') AS ClaimLocalCurrencySKey,
        COALESCE(CAST(DimCurrency_Base.CurrencySKey AS STRING), '-101') AS ClaimBaseCurrencySkey,
        COALESCE(DimDate_XchangeRate.Datekey, -101) AS ClaimXchangeRateDateKey,
        -- Amounts
        TMP.ClaimBilledAmountLocalCurrency,
        TMP.ClaimBilledAmountLocalCurrencyUSD,
        TMP.ClaimXchangeRate,
        TMP.ClaimBilledAmountBaseCurrency,
        TMP.ClaimBilledAmountBaseCurrencyUSD,
        TMP.ClaimIneligAmount,
        TMP.ClaimIneligAmountUSD,
        TMP.ClaimPrepaidAmount,
        TMP.ClaimPrepaidAmountUSD,
        TMP.ClaimCoInsuranceAmount,
        TMP.ClaimCoInsuranceAmountUSD,
        TMP.ClaimDeductibleAmount,
        TMP.ClaimDeductibleAmountUSD,
        TMP.ClaimAllowedAmount,
        TMP.ClaimAllowedAmountUSD,
        TMP.ClaimCoveredAmount,
        TMP.ClaimCoveredAmountUSD,
        TMP.ClaimNetCoveredAmount,
        TMP.ClaimNetCoveredAmountUSD,
        TMP.ClaimDiscountsAmount,
        TMP.ClaimDiscountsAmountUSD,
        TMP.ClaimNonCoveredAmount,
        TMP.ClaimNonCoveredAmountUSD,
        TMP.ClaimMemberLiabilityAmount,
        TMP.ClaimMemberLiabilityAmountUSD,
        TMP.ClaimProviderLiabilityAmount,
        TMP.ClaimProviderLiabilityAmountUSD,
        TMP.ClaimMemberPaymentAmount,
        TMP.ClaimMemberPaymentAmountUSD,
        COALESCE(CAST(DimCurrency_MemberPayment.CurrencySKey AS STRING), '-101') AS ClaimMemberPaymentCurrencySkey,
        TMP.ClaimMemberPaymentAmountCurrency,
        TMP.ClaimMemberPaymentAmountCurrencyUSD,
        TMP.ClaimMemberPaymentXchangeRate,
        TMP.ClaimProviderPaymentAmount,
        TMP.ClaimProviderPaymentAmountUSD,
        COALESCE(CAST(DimCurrency_ProviderPayment.CurrencySKey AS STRING), '-101') AS ClaimProviderPaymentCurrencySkey,
        TMP.ClaimProviderPaymentAmountCurrency,
        TMP.ClaimProviderPaymentAmountCurrencyUSD,
        TMP.ClaimProviderPaymentXchangeRate,
        TMP.ClaimCOBPrePaidAmount,
        TMP.ClaimCOBPrePaidAmountUSD,
        TMP.ClaimOtherPrePaidAmount,
        TMP.ClaimOtherPrePaidAmountUSD,
        COALESCE(CAST(DimDate_Processed.Datekey AS STRING), '-101') AS ClaimProcessedDateSkey,
        TMP.ControlVersion,
        COALESCE(CAST(DimStatus_ProcessedStatus.StatusSKey AS STRING), '-101') AS ClaimProcessedStatusSkey,
        COALESCE(CAST(DimClaimReason_Processed.ClaimReasonSKey AS STRING), '-101') AS ClaimProcessedReasonSkey,
        TMP.InOnNetwork,
        TMP.SubmissionInboundChannel,
        TMP.SubmissionTrackingNumber,
        TMP.SubmissionSenderType,
        TMP.SubmissionSenderEmailAddress,
        TMP.SubmissionSenderFullName,
        COALESCE(DimDate_SubmissionReceived.Datekey, -101) AS SubmissionReceivedDatekey,
        TMP.SubmissionIsComplement,
        TMP.ServiceNetwork,
        TMP.ClaimGap,
        COALESCE(CAST(DimReceivedMethod.ReceivedMethodSKey AS STRING), '-101') AS ClaimReceivedMethodSKey,
        TMP.PolicyFirstYear,
        TMP.PolicyFirstMonth,
        TMP.FirstClaim,
        TMP.IsDigital,
        COALESCE(CAST(DimServiceNetwork.ServiceNetworkSKey AS STRING), '-101') AS ServiceNetworkSKey,
        TMP.IsFastTrack,
        TMP.HTHFeePaid,
        TMP.HTHFeePercent,
        TMP.IsDeleted,
        SUBSTRING(TMP.ProviderNetworkClassName, 1, 200) AS ProviderNetworkClassName,
        0 AS SavingAmount,
        COALESCE(CAST(TMP.BrackedClaimKey AS STRING), '0') AS BrackedClaimKey,
        COALESCE(CAST(TMP.BrackedMemberKey AS STRING), '0') AS BrackedMemberKey,
        TMP.IncomebyinsuredIA AS `IncomebyinsuredIA`,
        TMP.SpendbyinsuredGA AS `SpendbyinsuredGA`,
        TMP.ClaimValidation,
        TMP.IsICU,
        TMP.IsEmergency,
        TMP.IsHospital_Emergency,
        TMP.IsChildbirths,
        TMP.IsCesareanSection,
        TMP.BatchLotName,
        TMP.BatchLotFromDate,
        TMP.TransactionLotReceivedDate,
        TMP.TransactionLotNumber,
        TMP.TransactionInvoiceNumber,
        TMP.LotNumber,
        TMP.IsAutomatic,
        TMP.FirstYearClaimExceptions,
        TMP.ClaimsOutliers,
        TMP.PolicyId,
        TMP.MemberId,
        -- ── NEW columns ──
        TMP.BaseWithHolding,
        TMP.BaseVAT,
        TMP.ThirdPartyId,
        SUBSTRING(COALESCE(TMP.LocalTeam, '_Non Defined LocalTeam'), 1, 100) AS LocalTeam,
        SUBSTRING(COALESCE(TMP.IsCOR, '_Not defined IsCOR'), 1, 100) AS IsCOR,
        TMP.ClaimCoInsuranceBISIAmount,
        TMP.ClaimCoInsuranceBISIAmountUSD,
        TMP.CoverageId,
        TMP.HigthCostClaim,
        COALESCE(DimDate_PaymentPosted.Datekey, -101) AS PaymentPostedDateKey,
        TMP.PaymentPostedDate,
        TMP.DeductibleOverrideCreatedOn,
        TMP.DeductibleOverrideCreatedBy,
        TMP.DeductibleOverrideCreatedByUserFullName,
        TMP.DeductibleOverrideApprovedOn,
        TMP.DeductibleOverrideApprovedBy,
        TMP.DeductibleOverrideApprovedByUserName,
        TMP.DeductibleOverrideReasonCode,
        TMP.DeductibleOverrideReasonDescription,
        TMP.UpdatedBy,
        TMP.ClaimType,
        -- ── NEW: Policy Dimension SKeys ──
        COALESCE(CAST(DimEntity_Policy.EntitySKey AS STRING), '-101') AS PolicyEntitySKey,
        COALESCE(CAST(DimGroup_Policy.GroupSKey AS STRING), '-101') AS PolicyGroupSKey,
        COALESCE(CAST(DimProduct_Policy.ProductSKey AS STRING), '-101') AS PolicyProductSKey,
        COALESCE(CAST(DimPlan_Policy.PlanSKey AS STRING), '-101') AS PolicyPlanSKey,
        COALESCE(CAST(DimBusinessMode_Policy.BusinessModeSKey AS STRING), '-101') AS PolicyBusinessModeSKey,
        COALESCE(CAST(DimBusinessType_Policy.BusinessTypeSKey AS STRING), '-101') AS PolicyBusinessTypeSKey,
        COALESCE(CAST(DimInsuranceBusiness_Policy.InsuranceBusinessSKey AS STRING), '-101') AS PolicyInsuranceBusinessSKey,
        COALESCE(CAST(DimCompany_Policy.CompanySKey AS STRING), '-101') AS PolicyCompanySKey,
        COALESCE(CAST(DimAgent_Policy.AgentSKey AS STRING), '-101') AS PolicyAgentSKey,
        COALESCE(CAST(DimRegion_Policy.RegionSKey AS STRING), '-101') AS PolicyRegionSkey,
        COALESCE(CAST(DimModeOfPayment_Policy.ModeOfPaymentSKey AS STRING), '-101') AS PolicyModeOfPaymentSKey,
        COALESCE(CAST(DimPaymentMethod_Policy.PaymentMethodSKey AS STRING), '-101') AS PolicyPaymentMethodSKey,
        COALESCE(CAST(DimReceivedMethod_Policy.ReceivedMethodSKey AS STRING), '-101') AS PolicyReceivedMethodSKey,
        COALESCE(CAST(DimFamilyType_Policy.FamilyTypeSKey AS STRING), '-101') AS PolicyFamilyTypeSKey,
        -- ── NEW: Policy Date Keys ──
        COALESCE(DimDate_PolicyIssue.Datekey, -101) AS PolicyIssueDateKey,
        COALESCE(DimDate_PolicyRenewal.Datekey, -101) AS PolicyRenewalDateKey,
        COALESCE(DimDate_PolicyAnniversary.Datekey, -101) AS PolicyAnniversaryDateKey,
        COALESCE(DimDate_PolicyAppReceived.Datekey, -101) AS PolicyApplicationReceivedDateKey,
        COALESCE(DimDate_PolicyEffective.Datekey, -101) AS PolicyEffectiveDateKey,
        COALESCE(DimDate_PolicyCancel.Datekey, -101) AS PolicyCancelDateKey,
        COALESCE(DimDate_PolicyDeathBenefit.Datekey, -101) AS PolicyDeathBenefitperiodDateKey,
        -- ── NEW: ThirdParty + SeverityBracket SKeys ──
        COALESCE(CAST(DimThirdParty.ThirdPartySKey AS STRING), '-101') AS PaymentThirdPartySKey,
        COALESCE(CAST(DimSeverityBracket.SeverityBracketSKey AS STRING), '-101') AS SeverityBracketSKey,
        -- ── NEW: VAT/Withholding USD conversions ──
        TMP.VATBaseCurrenty,
        TMP.VATUSDCurrenty,
        TMP.WithholdingBaseCurrenty,
        TMP.WithholdingUSDCurrenty,
        -- ── NEW: ClaimUpdatedByUser + Source ──
        TMP.ClaimUpdatedByUser,
        TMP.Source,
        TMP.IssuedDate,
        TMP.ActionDate,
        TMP.ActualRowFlag,
        TMP.EffectiveDate,
        TMP.LoadDate,
        -- ── Source IDs for InsertDataOrphan ──
        TMP.StatusId,
        TMP.ReasonId,
        TMP.MemberEligibilityId,
        TMP.BillingProviderId,
        TMP.ServiceProviderId,
        TMP.ScheduleId,
        TMP.DetailServiceIndicator,
        TMP.LineDetailServiceIndicator,
        TMP.ReportingServIndicatorId,
        TMP.DiagnosticId,
        TMP.ProcedureId,
        TMP.PlaceOfServiceId,
        TMP.TypeOfServiceId,
        TMP.PriorityId,
        TMP.GlobalLocationId,
        TMP.CountryId,
        TMP.StateId,
        TMP.CityId,
        TMP.AccumulatorScopeId,
        TMP.BillTypeId,
        TMP.PayTo,
        TMP.PayeeId,
        TMP.LocalCurrencyId,
        TMP.BaseCurrencyId,
        TMP.MemberPaymentCurrencyId,
        TMP.ProviderPaymentCurrencyId,
        TMP.ClaimProcessedStatusId,
        TMP.ClaimProcessedReasonId,
        TMP.ClaimReceivedMethodId,
        -- ── NEW: Policy dimension source IDs ──
        TMP.PolicyEntityId,
        TMP.PolicyGroupId,
        TMP.PolicyProductId,
        TMP.PolicyPlanId,
        TMP.PolicyBusinessModeId,
        TMP.PolicyBusinessTypeId,
        TMP.PolicyInsuranceBusinessId,
        TMP.PolicyCompanyId,
        TMP.PolicyAgentHierarchyId,
        TMP.PolicyRegionId,
        TMP.PolicyModeOfPaymentId,
        TMP.PolicyPaymentMethodId,
        TMP.PolicyReceivedMethodId,
        TMP.PolicyFamilyTypeId,
        CAST(TMP.StatusReasonDate AS DATE) AS StatusReasonDate,
        CAST(TMP.ServiceFromDate AS DATE) AS ServiceFromDate,
        CAST(TMP.ReceivedDate AS DATE) AS ReceivedDate,
        CAST(TMP.XchangeRateDate AS DATE) AS XchangeRateDate,
        CAST(TMP.ProcessedDate AS DATE) AS ProcessedDate,
        CAST(TMP.SubmissionReceivedDate AS DATE) AS SubmissionReceivedDate
    FROM Silver.TmpFactClaim TMP
    LEFT JOIN vw_DimStatus_bc DimStatus_ClaimStatus ON DimStatus_ClaimStatus.StatusId = TMP.StatusId
    LEFT JOIN vw_DimClaimReason_bc DimClaimReason ON DimClaimReason.ClaimReasonId = TMP.ReasonId
    LEFT JOIN vw_DimDate_bc DimDate_StatusReason ON DimDate_StatusReason.Date = TMP.StatusReasonDate
    LEFT JOIN vw_DimPolicyMember_bc DimMember_Claim ON DimMember_Claim.PolicyMemberEligibilityId = TMP.MemberEligibilityId AND DimMember_Claim.ToDate IS NULL
    LEFT JOIN vw_DimProvider_bc DimProvider_Billing ON DimProvider_Billing.ProviderId = TMP.BillingProviderId AND DimProvider_Billing.ProviderCategory = 'Billing'
    LEFT JOIN vw_DimProvider_bc DimProvider_Service ON DimProvider_Service.ProviderId = TMP.ServiceProviderId AND DimProvider_Service.ProviderCategory = 'Service'
    LEFT JOIN vw_DimSchedule_bc DimSchedule ON DimSchedule.ScheduleId = TMP.ScheduleId
    LEFT JOIN vw_DimDate_bc DimDate_ServiceFrom ON DimDate_ServiceFrom.Date = TMP.ServiceFromDate
    LEFT JOIN vw_DimServiceNetwork_bc DimServiceNetwork ON DimServiceNetwork.ServiceNetworkDescripcion = TMP.ServiceNetwork
    LEFT JOIN vw_DimServiceIndicator_bc DimServiceIndicator_Line ON DimServiceIndicator_Line.ServiceIndicatorId = TMP.LineDetailServiceIndicator
    LEFT JOIN vw_DimServiceIndicator_bc DimServiceIndicator_Reporting ON DimServiceIndicator_Reporting.ServiceIndicatorId = TMP.ReportingServIndicatorId
    LEFT JOIN vw_DimDiagnostic_bc DimDiagnostic ON DimDiagnostic.DiagnosticId = TMP.DiagnosticId
    LEFT JOIN vw_DimProcedure_bc DimProcedure ON DimProcedure.ProcedureId = TMP.ProcedureId
    LEFT JOIN vw_DimPlaceOfService_bc DimPlaceOfService ON DimPlaceOfService.PlaceOfServiceId = TMP.PlaceOfServiceId
    LEFT JOIN vw_DimTypeOfService_bc DimTypeOfService ON DimTypeOfService.TypeOfServiceId = TMP.TypeOfServiceId
    LEFT JOIN vw_DimPriority_bc DimPriority ON DimPriority.PriorityId = TMP.PriorityId
    LEFT JOIN vw_DimDate_bc DimDate_Received ON DimDate_Received.Date = TMP.ReceivedDate
    LEFT JOIN vw_DimGeography_bc DimGeography ON DimGeography.GlobalLocationId = TMP.GlobalLocationId AND DimGeography.CountryId = TMP.CountryId AND DimGeography.StateId = TMP.StateId AND DimGeography.CityId = TMP.CityId
    LEFT JOIN vw_DimAccumulatorScope_bc DimAccumulatorScope ON DimAccumulatorScope.AccumulatorScopeId = TMP.AccumulatorScopeId
    LEFT JOIN vw_DimBillType_bc DimBillType ON DimBillType.BillTypeId = TMP.BillTypeId
    LEFT JOIN vw_DimPayTo_bc DimPayTo ON DimPayTo.PayToId = TMP.PayTo
    LEFT JOIN vw_DimPayee_bc DimPayee ON DimPayee.PayeeId = TMP.PayeeId
    LEFT JOIN vw_DimPolicy_bc DimPolicy ON DimPolicy.PolicyId = TMP.PolicyId AND DimPolicy.ToDate IS NULL
    LEFT JOIN vw_DimCurrency_bc DimCurrency_Local ON DimCurrency_Local.CurrencyId = TMP.LocalCurrencyId
    LEFT JOIN vw_DimCurrency_bc DimCurrency_Base ON DimCurrency_Base.CurrencyId = TMP.BaseCurrencyId
    LEFT JOIN vw_DimCurrency_bc DimCurrency_MemberPayment ON DimCurrency_MemberPayment.CurrencyId = TMP.MemberPaymentCurrencyId
    LEFT JOIN vw_DimCurrency_bc DimCurrency_ProviderPayment ON DimCurrency_ProviderPayment.CurrencyId = TMP.ProviderPaymentCurrencyId
    LEFT JOIN vw_DimDate_bc DimDate_XchangeRate ON DimDate_XchangeRate.Date = TMP.XchangeRateDate
    LEFT JOIN vw_DimDate_bc DimDate_Processed ON DimDate_Processed.Date = TMP.ProcessedDate
    LEFT JOIN vw_DimStatus_bc DimStatus_ProcessedStatus ON DimStatus_ProcessedStatus.StatusId = TMP.ClaimProcessedStatusId
    LEFT JOIN vw_DimClaimReason_bc DimClaimReason_Processed ON DimClaimReason_Processed.ClaimReasonId = TMP.ClaimProcessedReasonId
    LEFT JOIN vw_DimDate_bc DimDate_SubmissionReceived ON DimDate_SubmissionReceived.Date = TMP.SubmissionReceivedDate
    LEFT JOIN vw_DimReceivedMethod_bc DimReceivedMethod ON DimReceivedMethod.ReceivedMethodId = TMP.ClaimReceivedMethodId
    LEFT JOIN vw_DimDate_bc DimDate_PaymentPosted ON DimDate_PaymentPosted.Date = TMP.PaymentPostedDate
    -- ── NEW: Policy Dimension JOINs ──
    LEFT JOIN vw_DimEntity_bc DimEntity_Policy ON DimEntity_Policy.EntityId = TMP.PolicyEntityId
    LEFT JOIN vw_DimGroup_bc DimGroup_Policy ON DimGroup_Policy.GroupId = TMP.PolicyGroupId AND DimGroup_Policy.ToDate IS NULL
    LEFT JOIN vw_DimProduct_bc DimProduct_Policy ON DimProduct_Policy.ProductId = TMP.PolicyProductId
    LEFT JOIN vw_DimPlan_bc DimPlan_Policy ON DimPlan_Policy.PlanId = TMP.PolicyPlanId
    LEFT JOIN vw_DimBusinessMode_bc DimBusinessMode_Policy ON DimBusinessMode_Policy.BusinessModeId = TMP.PolicyBusinessModeId
    LEFT JOIN vw_DimBusinessType_bc DimBusinessType_Policy ON DimBusinessType_Policy.BusinessTypeId = TMP.PolicyBusinessTypeId
    LEFT JOIN vw_DimInsuranceBusiness_bc DimInsuranceBusiness_Policy ON DimInsuranceBusiness_Policy.InsuranceBusinessId = TMP.PolicyInsuranceBusinessId AND DimInsuranceBusiness_Policy.ToDate IS NULL
    LEFT JOIN vw_DimCompany_bc DimCompany_Policy ON DimCompany_Policy.CompanyId = TMP.PolicyCompanyId
    LEFT JOIN vw_DimAgent_bc DimAgent_Policy ON DimAgent_Policy.AgentHierarchyId = TMP.PolicyAgentHierarchyId
    LEFT JOIN vw_DimRegion_bc DimRegion_Policy ON DimRegion_Policy.RegionId = TMP.PolicyRegionId
    LEFT JOIN vw_DimModeOfPayment_bc DimModeOfPayment_Policy ON DimModeOfPayment_Policy.ModeOfPaymentId = TMP.PolicyModeOfPaymentId
    LEFT JOIN vw_DimPaymentMethod_bc DimPaymentMethod_Policy ON DimPaymentMethod_Policy.PaymentMethodId = TMP.PolicyPaymentMethodId
    LEFT JOIN vw_DimReceivedMethod_bc DimReceivedMethod_Policy ON DimReceivedMethod_Policy.ReceivedMethodId = TMP.PolicyReceivedMethodId
    LEFT JOIN vw_DimFamilyType_bc DimFamilyType_Policy ON DimFamilyType_Policy.FamilyTypeId = TMP.PolicyFamilyTypeId
    -- ── NEW: Policy Date JOINs ──
    LEFT JOIN vw_DimDate_bc DimDate_PolicyIssue ON DimDate_PolicyIssue.Date = TMP.PolicyIssueDateRaw
    LEFT JOIN vw_DimDate_bc DimDate_PolicyRenewal ON DimDate_PolicyRenewal.Date = TMP.PolicyRenewalDateRaw
    LEFT JOIN vw_DimDate_bc DimDate_PolicyAnniversary ON DimDate_PolicyAnniversary.Date = TMP.PolicyAnniversaryDateRaw
    LEFT JOIN vw_DimDate_bc DimDate_PolicyAppReceived ON DimDate_PolicyAppReceived.Date = TMP.PolicyApplicationReceivedDateRaw
    LEFT JOIN vw_DimDate_bc DimDate_PolicyEffective ON DimDate_PolicyEffective.Date = TMP.PolicyEffectiveDateRaw
    LEFT JOIN vw_DimDate_bc DimDate_PolicyCancel ON DimDate_PolicyCancel.Date = TMP.PolicyCancelDateRaw
    LEFT JOIN vw_DimDate_bc DimDate_PolicyDeathBenefit ON DimDate_PolicyDeathBenefit.Date = TMP.PolicyDeathBenefitPeriodDateRaw
    -- ── NEW: ThirdParty + SeverityBracket ──
    LEFT JOIN vw_DimThirdParty_bc DimThirdParty ON DimThirdParty.ThirdPartyId = TMP.ThirdPartyId
    LEFT JOIN vw_DimSeverityBracket_bc DimSeverityBracket ON DimSeverityBracket.ClaimHeaderId = TMP.HeaderId AND DimSeverityBracket.MemberId = TMP.MemberId
""")

df_final.createOrReplaceTempView("vwTmp_RegistrosFactClaim")
print(f"[OK] vwTmp_RegistrosFactClaim created with broadcast dimension joins + source IDs")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 36, Finished, Available, Finished, False)

[OK] vwTmp_RegistrosFactClaim created with broadcast dimension joins + source IDs


In [35]:
%run nb_Utils

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 38, Finished, Available, Finished, True)

In [36]:
#InsertDataOrphan("bgla_FactClaim")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 39, Finished, Available, Finished, False)

In [37]:
# ============================================================
# MERGE DELETE + INSERT (consolidated)
# El MERGE ON verifica TODAS las keys de la fact table
# para evitar duplicados y eliminar solo filas exactas.
# Luego inserta los registros frescos con dedup por rn=1
# ============================================================

# ── Step 1: MERGE DELETE (all keys verified) ──
spark.sql("""
    MERGE INTO Gold.bgla_FactClaim AS T
    USING vwTmp_RegistrosFactClaim AS TMP
    ON  T.ClaimDetailId = TMP.ClaimDetailId
    AND T.ClaimLineDetailId = TMP.ClaimLineDetailId
    AND T.ClaimVersion = TMP.ClaimVersion
    AND T.ClaimHeaderId = TMP.ClaimHeaderId
    AND T.ClaimNumber = TMP.ClaimNumber
    AND COALESCE(T.ClaimStatusSKey, '-101') = COALESCE(TMP.ClaimStatusSKey, '-101')
    AND COALESCE(T.ClaimReasonSKey, '-101') = COALESCE(TMP.ClaimReasonSKey, '-101')
    AND COALESCE(T.ClaimStatusReasonDatekey, -101) = COALESCE(TMP.ClaimStatusReasonDatekey, -101)
    AND COALESCE(T.ClaimMemberSKey, '-101') = COALESCE(TMP.ClaimMemberSKey, '-101')
    AND COALESCE(T.ClaimBillingProviderSkey, '-101') = COALESCE(TMP.ClaimBillingProviderSkey, '-101')
    AND COALESCE(T.ClaimServiceProviderSKey, '-101') = COALESCE(TMP.ClaimServiceProviderSKey, '-101')
    AND COALESCE(T.ClaimScheduleSKey, '-101') = COALESCE(TMP.ClaimScheduleSKey, '-101')
    AND COALESCE(T.ClaimServiceFromDateKey, -101) = COALESCE(TMP.ClaimServiceFromDateKey, -101)
    AND COALESCE(T.ClaimNetworkSkey, '-101') = COALESCE(TMP.ClaimNetworkSkey, '-101')
    AND COALESCE(T.ClaimLineDetailServiceindicatorSkey, '-101') = COALESCE(TMP.ClaimLineDetailServiceindicatorSkey, '-101')
    AND COALESCE(T.ClaimReportingServiceindicatorSkey, '-101') = COALESCE(TMP.ClaimReportingServiceindicatorSkey, '-101')
    AND COALESCE(T.ClaimPrimaryDiagnosticSkey, '-101') = COALESCE(TMP.ClaimPrimaryDiagnosticSkey, '-101')
    AND COALESCE(T.ClaimPrimaryProcedureSkey, '-101') = COALESCE(TMP.ClaimPrimaryProcedureSkey, '-101')
    AND COALESCE(T.ClaimPlaceOfServiceSKey, '-101') = COALESCE(TMP.ClaimPlaceOfServiceSKey, '-101')
    AND COALESCE(T.ClaimTypeOfServiceSKey, '-101') = COALESCE(TMP.ClaimTypeOfServiceSKey, '-101')
    AND COALESCE(T.ClaimPrioritySkey, '-101') = COALESCE(TMP.ClaimPrioritySkey, '-101')
    AND COALESCE(T.ClaimReceivedDateKey, -101) = COALESCE(TMP.ClaimReceivedDateKey, -101)
    AND COALESCE(T.ClaimGeographySKey, '-101') = COALESCE(TMP.ClaimGeographySKey, '-101')
    AND COALESCE(T.ClaimAccumulatorScopeSKey, '-101') = COALESCE(TMP.ClaimAccumulatorScopeSKey, '-101')
    AND COALESCE(T.ClaimBillTypeSKey, '-101') = COALESCE(TMP.ClaimBillTypeSKey, '-101')
    AND COALESCE(T.ClaimPayToSKey, '-101') = COALESCE(TMP.ClaimPayToSKey, '-101')
    AND COALESCE(T.ClaimPayeeSkey, '-101') = COALESCE(TMP.ClaimPayeeSkey, '-101')
    AND COALESCE(T.PolicySKey, '-101') = COALESCE(TMP.PolicySKey, '-101')
    AND COALESCE(T.ClaimLocalCurrencySKey, '-101') = COALESCE(TMP.ClaimLocalCurrencySKey, '-101')
    AND COALESCE(T.ClaimBaseCurrencySkey, '-101') = COALESCE(TMP.ClaimBaseCurrencySkey, '-101')
    AND COALESCE(T.ClaimXchangeRateDateKey, -101) = COALESCE(TMP.ClaimXchangeRateDateKey, -101)
    AND COALESCE(T.ClaimProcessedDateSkey, '-101') = COALESCE(TMP.ClaimProcessedDateSkey, '-101')
    AND COALESCE(T.ClaimProcessedStatusSkey, '-101') = COALESCE(TMP.ClaimProcessedStatusSkey, '-101')
    AND COALESCE(T.ClaimProcessedReasonSkey, '-101') = COALESCE(TMP.ClaimProcessedReasonSkey, '-101')
    AND COALESCE(T.ClaimReceivedMethodSKey, '-101') = COALESCE(TMP.ClaimReceivedMethodSKey, '-101')
    AND COALESCE(T.ServiceNetworkSKey, '-101') = COALESCE(TMP.ServiceNetworkSKey, '-101')
    AND COALESCE(T.ClaimMemberPaymentCurrencySkey, '-101') = COALESCE(TMP.ClaimMemberPaymentCurrencySkey, '-101')
    AND COALESCE(T.ClaimProviderPaymentCurrencySkey, '-101') = COALESCE(TMP.ClaimProviderPaymentCurrencySkey, '-101')
    AND COALESCE(T.SubmissionReceivedDatekey, -101) = COALESCE(TMP.SubmissionReceivedDatekey, -101)
    AND COALESCE(T.PaymentPostedDateKey, -101) = COALESCE(TMP.PaymentPostedDateKey, -101)
    AND COALESCE(T.PolicyEntitySKey, '-101') = COALESCE(TMP.PolicyEntitySKey, '-101')
    AND COALESCE(T.PolicyGroupSKey, '-101') = COALESCE(TMP.PolicyGroupSKey, '-101')
    AND COALESCE(T.PolicyProductSKey, '-101') = COALESCE(TMP.PolicyProductSKey, '-101')
    AND COALESCE(T.PolicyPlanSKey, '-101') = COALESCE(TMP.PolicyPlanSKey, '-101')
    AND COALESCE(T.PolicyInsuranceBusinessSKey, '-101') = COALESCE(TMP.PolicyInsuranceBusinessSKey, '-101')
    AND COALESCE(T.PolicyCompanySKey, '-101') = COALESCE(TMP.PolicyCompanySKey, '-101')
    AND COALESCE(T.PolicyAgentSKey, '-101') = COALESCE(TMP.PolicyAgentSKey, '-101')
    AND COALESCE(T.PolicyRegionSkey, '-101') = COALESCE(TMP.PolicyRegionSkey, '-101')
    AND COALESCE(T.PaymentThirdPartySKey, '-101') = COALESCE(TMP.PaymentThirdPartySKey, '-101')
    AND COALESCE(T.SeverityBracketSKey, '-101') = COALESCE(TMP.SeverityBracketSKey, '-101')
    WHEN MATCHED THEN DELETE
""")
print("[OK] MERGE DELETE completed (all keys verified)")

# ── Step 2: INSERT fresh delta records (dedup by rn=1) ──
spark.sql("""
    INSERT INTO Gold.bgla_FactClaim
    (FactClaimSKey, ClaimDetailId, LogId, ClaimLineDetailId, ClaimLineSequenceNumber, ClaimVersion,
     ClaimReferralHeaderId, ClaimUUID, ClaimHeaderId, ClaimNumber,
     ClaimStatusSKey, ClaimReasonSKey, ClaimStatusReasonDatekey,
     ClaimMemberSKey, ClaimBillingProviderSkey, ClaimServiceProviderSKey, ClaimScheduleSKey,
     ClaimServiceFromDateKey, ClaimServiceToDate, ClaimNetworkSkey,
     ClaimLineDetailServiceindicatorSkey, ClaimReportingServiceindicatorSkey,
     ClaimPrimaryDiagnosticSkey, ClaimPrimaryProcedureSkey,
     ClaimPlaceOfServiceSKey, ClaimTypeOfServiceSKey, ClaimPrioritySkey, ClaimReceivedDateKey,
     ClaimGeographySKey, ClaimAccumulatorScopeSKey, ClaimBillTypeSKey, ClaimPayToSKey, ClaimPayeeSkey,
     ClaimAccountNumber, MemberPPA, ProviderPPA, PolicySKey,
     ClaimReferenceNumber, ClaimPenalty, ClaimQuantity,
     ClaimLocalCurrencySKey, ClaimBaseCurrencySkey, ClaimXchangeRateDateKey,
     ClaimBilledAmountLocalCurrency, ClaimBilledAmountLocalCurrencyUSD,
     ClaimXchangeRate, ClaimBilledAmountBaseCurrency, ClaimBilledAmountBaseCurrencyUSD,
     ClaimIneligAmount, ClaimIneligAmountUSD, ClaimPrepaidAmount, ClaimPrepaidAmountUSD,
     ClaimCoInsuranceAmount, ClaimCoInsuranceAmountUSD, ClaimDeductibleAmount, ClaimDeductibleAmountUSD,
     ClaimAllowedAmount, ClaimAllowedAmountUSD, ClaimCoveredAmount, ClaimCoveredAmountUSD,
     ClaimNetCoveredAmount, ClaimNetCoveredAmountUSD, ClaimDiscountsAmount, ClaimDiscountsAmountUSD,
     ClaimNonCoveredAmount, ClaimNonCoveredAmountUSD,
     ClaimMemberLiabilityAmount, ClaimMemberLiabilityAmountUSD,
     ClaimProviderLiabilityAmount, ClaimProviderLiabilityAmountUSD,
     ClaimMemberPaymentAmount, ClaimMemberPaymentAmountUSD,
     ClaimMemberPaymentCurrencySkey, ClaimMemberPaymentAmountCurrency, ClaimMemberPaymentAmountCurrencyUSD,
     ClaimMemberPaymentXchangeRate, ClaimProviderPaymentAmount, ClaimProviderPaymentAmountUSD,
     ClaimProviderPaymentCurrencySkey, ClaimProviderPaymentAmountCurrency, ClaimProviderPaymentAmountCurrencyUSD,
     ClaimProviderPaymentXchangeRate, ClaimCOBPrePaidAmount, ClaimCOBPrePaidAmountUSD,
     ClaimOtherPrePaidAmount, ClaimOtherPrePaidAmountUSD,
     ClaimProcessedDateSkey, ControlVersion, ClaimProcessedStatusSkey, ClaimProcessedReasonSkey,
     InOnNetwork, SubmissionInboundChannel, SubmissionTrackingNumber, SubmissionSenderType,
     SubmissionSenderEmailAddress, SubmissionSenderFullName,
     SubmissionReceivedDatekey, SubmissionIsComplement, ServiceNetwork,
     ClaimGap, ClaimReceivedMethodSKey, PolicyFirstYear, PolicyFirstMonth, FirstClaim,
     IsDigital, ServiceNetworkSKey, IsFastTrack, HTHFeePaid, HTHFeePercent,
     IsDeleted, ProviderNetworkClassName, SavingAmount, BrackedClaimKey, BrackedMemberKey,
     `IncomebyinsuredIA`, `SpendbyinsuredGA`, ClaimValidation,
     IsICU, IsEmergency, IsHospital_Emergency, IsChildbirths, IsCesareanSection,
     BatchLotName, BatchLotFromDate, TransactionLotReceivedDate, TransactionLotNumber,
     TransactionInvoiceNumber, LotNumber, IsAutomatic, FirstYearClaimExceptions,
     ClaimsOutliers, PolicyId, MemberId,
     BaseWithHolding, BaseVAT, ThirdPartyId, LocalTeam, IsCOR,
     ClaimCoInsuranceBISIAmount, ClaimCoInsuranceBISIAmountUSD, CoverageId, HigthCostClaim,
     PaymentPostedDateKey, PaymentPostedDate,
     DeductibleOverrideCreatedOn, DeductibleOverrideCreatedBy, DeductibleOverrideCreatedByUserFullName,
     DeductibleOverrideApprovedOn, DeductibleOverrideApprovedBy, DeductibleOverrideApprovedByUserName,
     DeductibleOverrideReasonCode, DeductibleOverrideReasonDescription,
     UpdatedBy, ClaimType, IssuedDate, ActionDate,
     PolicyEntitySKey, PolicyGroupSKey, PolicyProductSKey, PolicyPlanSKey,
     PolicyBusinessModeSKey, PolicyBusinessTypeSKey, PolicyInsuranceBusinessSKey,
     PolicyCompanySKey, PolicyAgentSKey, PolicyRegionSkey,
     PolicyModeOfPaymentSKey, PolicyPaymentMethodSKey, PolicyReceivedMethodSKey, PolicyFamilyTypeSKey,
     PolicyIssueDateKey, PolicyRenewalDateKey, PolicyAnniversaryDateKey,
     PolicyApplicationReceivedDateKey, PolicyEffectiveDateKey, PolicyCancelDateKey, PolicyDeathBenefitperiodDateKey,
     PaymentThirdPartySKey, SeverityBracketSKey,
     VATBaseCurrenty, VATUSDCurrenty, WithholdingBaseCurrenty, WithholdingUSDCurrenty,
     ClaimUpdatedByUser, Source,
     ActualRowFlag, EffectiveDate, LoadDate)
    SELECT
     FactClaimSKey, ClaimDetailId, LogId, ClaimLineDetailId, ClaimLineSequenceNumber, ClaimVersion,
     ClaimReferralHeaderId, ClaimUUID, ClaimHeaderId, ClaimNumber,
     ClaimStatusSKey, ClaimReasonSKey, ClaimStatusReasonDatekey,
     ClaimMemberSKey, ClaimBillingProviderSkey, ClaimServiceProviderSKey, ClaimScheduleSKey,
     ClaimServiceFromDateKey, ClaimServiceToDate, ClaimNetworkSkey,
     ClaimLineDetailServiceindicatorSkey, ClaimReportingServiceindicatorSkey,
     ClaimPrimaryDiagnosticSkey, ClaimPrimaryProcedureSkey,
     ClaimPlaceOfServiceSKey, ClaimTypeOfServiceSKey, ClaimPrioritySkey, ClaimReceivedDateKey,
     ClaimGeographySKey, ClaimAccumulatorScopeSKey, ClaimBillTypeSKey, ClaimPayToSKey, ClaimPayeeSkey,
     ClaimAccountNumber, MemberPPA, ProviderPPA, PolicySKey,
     ClaimReferenceNumber, ClaimPenalty, ClaimQuantity,
     ClaimLocalCurrencySKey, ClaimBaseCurrencySkey, ClaimXchangeRateDateKey,
     ClaimBilledAmountLocalCurrency, ClaimBilledAmountLocalCurrencyUSD,
     ClaimXchangeRate, ClaimBilledAmountBaseCurrency, ClaimBilledAmountBaseCurrencyUSD,
     ClaimIneligAmount, ClaimIneligAmountUSD, ClaimPrepaidAmount, ClaimPrepaidAmountUSD,
     ClaimCoInsuranceAmount, ClaimCoInsuranceAmountUSD, ClaimDeductibleAmount, ClaimDeductibleAmountUSD,
     ClaimAllowedAmount, ClaimAllowedAmountUSD, ClaimCoveredAmount, ClaimCoveredAmountUSD,
     ClaimNetCoveredAmount, ClaimNetCoveredAmountUSD, ClaimDiscountsAmount, ClaimDiscountsAmountUSD,
     ClaimNonCoveredAmount, ClaimNonCoveredAmountUSD,
     ClaimMemberLiabilityAmount, ClaimMemberLiabilityAmountUSD,
     ClaimProviderLiabilityAmount, ClaimProviderLiabilityAmountUSD,
     ClaimMemberPaymentAmount, ClaimMemberPaymentAmountUSD,
     ClaimMemberPaymentCurrencySkey, ClaimMemberPaymentAmountCurrency, ClaimMemberPaymentAmountCurrencyUSD,
     ClaimMemberPaymentXchangeRate, ClaimProviderPaymentAmount, ClaimProviderPaymentAmountUSD,
     ClaimProviderPaymentCurrencySkey, ClaimProviderPaymentAmountCurrency, ClaimProviderPaymentAmountCurrencyUSD,
     ClaimProviderPaymentXchangeRate, ClaimCOBPrePaidAmount, ClaimCOBPrePaidAmountUSD,
     ClaimOtherPrePaidAmount, ClaimOtherPrePaidAmountUSD,
     ClaimProcessedDateSkey, ControlVersion, ClaimProcessedStatusSkey, ClaimProcessedReasonSkey,
     InOnNetwork, SubmissionInboundChannel, SubmissionTrackingNumber, SubmissionSenderType,
     SubmissionSenderEmailAddress, SubmissionSenderFullName,
     SubmissionReceivedDatekey, SubmissionIsComplement, ServiceNetwork,
     ClaimGap, ClaimReceivedMethodSKey, PolicyFirstYear, PolicyFirstMonth, FirstClaim,
     IsDigital, ServiceNetworkSKey, IsFastTrack, HTHFeePaid, HTHFeePercent,
     IsDeleted, ProviderNetworkClassName, SavingAmount, BrackedClaimKey, BrackedMemberKey,
     `IncomebyinsuredIA`, `SpendbyinsuredGA`, ClaimValidation,
     IsICU, IsEmergency, IsHospital_Emergency, IsChildbirths, IsCesareanSection,
     BatchLotName, BatchLotFromDate, TransactionLotReceivedDate, TransactionLotNumber,
     TransactionInvoiceNumber, LotNumber, IsAutomatic, FirstYearClaimExceptions,
     ClaimsOutliers, PolicyId, MemberId,
     BaseWithHolding, BaseVAT, ThirdPartyId, LocalTeam, IsCOR,
     ClaimCoInsuranceBISIAmount, ClaimCoInsuranceBISIAmountUSD, CoverageId, HigthCostClaim,
     PaymentPostedDateKey, PaymentPostedDate,
     DeductibleOverrideCreatedOn, DeductibleOverrideCreatedBy, DeductibleOverrideCreatedByUserFullName,
     DeductibleOverrideApprovedOn, DeductibleOverrideApprovedBy, DeductibleOverrideApprovedByUserName,
     DeductibleOverrideReasonCode, DeductibleOverrideReasonDescription,
     UpdatedBy, ClaimType, IssuedDate, ActionDate,
     PolicyEntitySKey, PolicyGroupSKey, PolicyProductSKey, PolicyPlanSKey,
     PolicyBusinessModeSKey, PolicyBusinessTypeSKey, PolicyInsuranceBusinessSKey,
     PolicyCompanySKey, PolicyAgentSKey, PolicyRegionSkey,
     PolicyModeOfPaymentSKey, PolicyPaymentMethodSKey, PolicyReceivedMethodSKey, PolicyFamilyTypeSKey,
     PolicyIssueDateKey, PolicyRenewalDateKey, PolicyAnniversaryDateKey,
     PolicyApplicationReceivedDateKey, PolicyEffectiveDateKey, PolicyCancelDateKey, PolicyDeathBenefitperiodDateKey,
     PaymentThirdPartySKey, SeverityBracketSKey,
     VATBaseCurrenty, VATUSDCurrenty, WithholdingBaseCurrenty, WithholdingUSDCurrenty,
     ClaimUpdatedByUser, Source,
     ActualRowFlag, EffectiveDate, LoadDate
    FROM (
      SELECT *,
        ROW_NUMBER() OVER (PARTITION BY ClaimDetailId, ClaimLineDetailId, ClaimVersion ORDER BY LoadDate DESC) AS rn
      FROM vwTmp_RegistrosFactClaim
    ) ranked
    WHERE rn = 1
""")

row_count = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaim").first()["cnt"]
print(f"[OK] MERGE DELETE + INSERT completed. Gold.bgla_FactClaim total rows: {row_count}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 40, Finished, Available, Finished, False)

[OK] MERGE DELETE completed (all keys verified)
[OK] MERGE DELETE + INSERT completed. Gold.bgla_FactClaim total rows: 12945318


In [38]:
# ============================================================
# DEDUP ActualRowFlag + EffectiveDate (consolidated)
# Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims (lines 17-41)
# 1. Marca ActualRowFlag = 0 en filas no-actuales (DENSE_RANK)
# 2. Actualiza EffectiveDate en las filas actuales
# ============================================================

# ── Step 1: ActualRowFlag dedup ──
spark.sql("""
    MERGE INTO Gold.bgla_FactClaim F
    USING (
      SELECT t.FactClaimSKey
      FROM (
        SELECT F.FactClaimSKey,
          DENSE_RANK() OVER (PARTITION BY F.ClaimDetailId, F.ClaimLineDetailId ORDER BY F.LoadDate DESC) AS Fila,
          F.ActualRowFlag
        FROM Gold.bgla_FactClaim F
      ) t
      WHERE t.Fila > 1 AND t.ActualRowFlag <> 0
    ) P
    ON F.FactClaimSKey = P.FactClaimSKey
    WHEN MATCHED THEN UPDATE SET F.ActualRowFlag = 0
""")
print("[OK] ActualRowFlag dedup completed")

# ── Step 2: EffectiveDate update ──
spark.sql("""
    UPDATE Gold.bgla_FactClaim
    SET EffectiveDate = current_date()
    WHERE ActualRowFlag = 1
""")
print("[OK] EffectiveDate updated for active records")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 41, Finished, Available, Finished, False)

[OK] ActualRowFlag dedup completed
[OK] EffectiveDate updated for active records


In [39]:
# ============================================================
# IsDeleted mark
# Marca IsDeleted = 1 en headers cuyo Status = 'DELETED' o Reason LIKE '%DELETED%'
# ============================================================
spark.sql("""
    MERGE INTO Gold.bgla_FactClaim F
    USING (
        SELECT DISTINCT F.ClaimHeaderId
        FROM Gold.bgla_FactClaim F
        INNER JOIN Gold.bgla_DimStatus st
            ON CAST(st.StatusSKey AS STRING) = F.ClaimStatusSKey
        INNER JOIN Gold.bgla_DimClaimReason cr
            ON CAST(cr.ClaimReasonSKey AS STRING) = F.ClaimReasonSKey
        WHERE st.StatusDescription = 'DELETED'
           OR cr.ClaimReasonName LIKE '%DELETED%'
    ) D
    ON F.ClaimHeaderId = D.ClaimHeaderId
    WHEN MATCHED THEN UPDATE SET F.IsDeleted = 1
""")
print("[OK] IsDeleted mark applied")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 42, Finished, Available, Finished, False)

[OK] IsDeleted mark applied


In [40]:
# ============================================================
# ClaimsOutliers mark
# Marca ClaimsOutliers = 1 en headers donde suma de BilledAmountBaseCurrencyUSD
# está fuera del rango [MIN_OUTLIER, MAX_OUTLIER]
# ============================================================
spark.sql(f"""
    MERGE INTO Gold.bgla_FactClaim F
    USING (
        SELECT ClaimHeaderId
        FROM Gold.bgla_FactClaim
        WHERE ActualRowFlag = 1
        GROUP BY ClaimHeaderId
        HAVING SUM(ClaimBilledAmountBaseCurrencyUSD) NOT BETWEEN {MIN_OUTLIER} AND {MAX_OUTLIER}
    ) O
    ON F.ClaimHeaderId = O.ClaimHeaderId
    WHEN MATCHED THEN UPDATE SET F.ClaimsOutliers = 1
""")
print(f"[OK] ClaimsOutliers mark applied (range: {MIN_OUTLIER} - {MAX_OUTLIER})")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 43, Finished, Available, Finished, False)

[OK] ClaimsOutliers mark applied (range: 0 - 99999999)


In [41]:
# ============================================================
# DiseasesAssociated
# Trunca y recarga Gold.bgla_FactClaimMemberDiseasesAssociated
# con la agregación de enfermedades por MemberId
# Equivale a #TDiseases + INSERT del SP
# ============================================================
try:
    spark.sql("TRUNCATE TABLE Gold.bgla_FactClaimMemberDiseasesAssociated")

    spark.sql("""
        INSERT INTO Gold.bgla_FactClaimMemberDiseasesAssociated (MemberId, DiseasesAssociated)
        SELECT
            MemberId,
            SUBSTRING(CONCAT_WS(',', COLLECT_SET(DiseasesAssociated)), 1, 500) AS DiseasesAssociated
        FROM (
            SELECT DISTINCT FC.MemberId, D.DiseasesAssociated
            FROM Gold.bgla_FactClaim FC
            INNER JOIN Gold.bgla_DimDiagnostic D
                ON CAST(D.DiagnosticSKey AS STRING) = FC.ClaimPrimaryDiagnosticSkey
            WHERE D.DiseasesAssociated <> 'Not Diseases Associated'
        ) base
        GROUP BY MemberId
    """)

    cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaimMemberDiseasesAssociated").first()["cnt"]
    print(f"[OK] DiseasesAssociated loaded: {cnt} rows")
except Exception as e:
    print(f"[WARN] DiseasesAssociated skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 44, Finished, Available, Finished, False)

[WARN] DiseasesAssociated skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Gold`.`bgla_FactClaimMemberDiseasesAssociated` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 0;
'TruncateCommand
+- 'UnresolvedTable [Gold, bgla_FactClaimMemberDiseasesAssociated], TRUNCATE



In [42]:
# ============================================================
# Readmission by Procedure
# Calcula Day Of Readmission Procedure y CaseReadmissionProcedure
# Particionado por PolicyId, MemberId, BillingProvider, Diagnostic, Procedure
# Trunca FactClaimMemberCaseReadmission y carga con datos de Procedure
# ============================================================
try:
    spark.sql("TRUNCATE TABLE Gold.bgla_FactClaimMemberCaseReadmission")

    # ── Base: agrupación por Procedure ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ReadmissionProcedureBase AS
        SELECT
            FC.ClaimHeaderId,
            Pol.PolicyId,
            FC.MemberId,
            FC.ClaimBillingProviderSkey,
            FC.ClaimPrimaryDiagnosticSkey,
            FC.ClaimPrimaryProcedureSkey,
            MIN(CAST(CONCAT(
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 1, 4), '-',
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 5, 2), '-',
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 7, 2)
            ) AS DATE)) AS ClaimServiceFromDate,
            MAX(FC.ClaimServiceToDate) AS ClaimServiceToDate
        FROM Gold.bgla_FactClaim FC
        INNER JOIN Gold.bgla_DimPolicy Pol ON FC.PolicySKey = CAST(Pol.PolicySKey AS STRING)
        INNER JOIN Gold.bgla_DimClaimReason dRS ON FC.ClaimReasonSKey = CAST(dRS.ClaimReasonSKey AS STRING)
        INNER JOIN Gold.bgla_DimStatus DS ON FC.ClaimStatusSKey = CAST(DS.StatusSKey AS STRING)
        WHERE FC.ControlVersion = 'Current Version'
          AND dRS.ClaimReasonName <> 'DELETED'
          AND DS.StatusTypeName = 'Claim Status'
          AND DS.StatusDescription = 'Processed'
          AND dRS.ClaimReasonName <> 'DENIED'
          AND FC.ClaimServiceFromDateKey > 19000101
        GROUP BY FC.ClaimHeaderId, Pol.PolicyId, FC.MemberId,
                 FC.ClaimBillingProviderSkey, FC.ClaimPrimaryDiagnosticSkey, FC.ClaimPrimaryProcedureSkey
    """)

    # ── Day of Readmission Procedure (con LAG, eliminando negativos y recalculando) ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ReadmissionProcedureDay AS
        SELECT ClaimHeaderId, ClaimServiceFromDate, ClaimServiceToDate,
               ClaimBillingProviderSkey, ClaimPrimaryDiagnosticSkey, ClaimPrimaryProcedureSkey,
               PolicyId, MemberId,
               DATEDIFF(
                   ClaimServiceFromDate,
                   LAG(ClaimServiceToDate, 1) OVER (
                       PARTITION BY PolicyId, MemberId, ClaimBillingProviderSkey,
                                    ClaimPrimaryDiagnosticSkey, ClaimPrimaryProcedureSkey
                       ORDER BY ClaimServiceFromDate, ClaimHeaderId
                   )
               ) AS DayOfReadmissionProcedure
        FROM (
            -- Recalculate after removing negatives
            SELECT ClaimHeaderId, ClaimServiceFromDate, ClaimServiceToDate,
                   ClaimBillingProviderSkey, ClaimPrimaryDiagnosticSkey, ClaimPrimaryProcedureSkey,
                   PolicyId, MemberId
            FROM (
                SELECT *,
                    DATEDIFF(
                        ClaimServiceFromDate,
                        LAG(ClaimServiceToDate, 1) OVER (
                            PARTITION BY PolicyId, MemberId, ClaimBillingProviderSkey,
                                         ClaimPrimaryDiagnosticSkey, ClaimPrimaryProcedureSkey
                            ORDER BY ClaimServiceFromDate, ClaimHeaderId
                        )
                    ) AS day_calc
                FROM vw_ReadmissionProcedureBase
            ) t1
            WHERE day_calc IS NULL OR day_calc >= 0
        ) t2
    """)

    # ── INSERT Readmission by Procedure ──
    spark.sql("""
        INSERT INTO Gold.bgla_FactClaimMemberCaseReadmission
        (ClaimHeaderId, ClaimServiceToDate, ClaimServiceFromDate, ClaimBillingProviderSkey,
         PolicyId, MemberId, ClaimPrimaryDiagnosticSkey,
         `Day Of Readmission`, CaseReadmission,
         ClaimPrimaryProcedureSkey, `Day Of Readmission Procedure`,
         `Day Of Readmission Provider`, CaseReadmissionProcedure, CaseReadmissionProvider,
         `Readmission by Procedure`, `Readmission by Diagnostic`, `Readmission by Provider`)
        SELECT
            RP.ClaimHeaderId,
            RP.ClaimServiceToDate,
            RP.ClaimServiceFromDate,
            RP.ClaimBillingProviderSkey,
            RP.PolicyId,
            RP.MemberId,
            RP.ClaimPrimaryDiagnosticSkey,
            0 AS `Day Of Readmission`,
            0 AS CaseReadmission,
            RP.ClaimPrimaryProcedureSkey,
            RP.DayOfReadmissionProcedure AS `Day Of Readmission Procedure`,
            0 AS `Day Of Readmission Provider`,
            CASE WHEN RP.DayOfReadmissionProcedure BETWEEN 0 AND 30 THEN 1 ELSE 0 END AS CaseReadmissionProcedure,
            0 AS CaseReadmissionProvider,
            1 AS `Readmission by Procedure`,
            1 AS `Readmission by Diagnostic`,
            1 AS `Readmission by Provider`
        FROM vw_ReadmissionProcedureDay RP
        WHERE RP.DayOfReadmissionProcedure IS NOT NULL
    """)

    cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaimMemberCaseReadmission").first()["cnt"]
    print(f"[OK] Readmission by Procedure loaded: {cnt} rows")
except Exception as e:
    print(f"[WARN] Readmission by Procedure skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 45, Finished, Available, Finished, False)

[WARN] Readmission by Procedure skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Gold`.`bgla_FactClaimMemberCaseReadmission` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 0;
'TruncateCommand
+- 'UnresolvedTable [Gold, bgla_FactClaimMemberCaseReadmission], TRUNCATE



In [43]:
# ============================================================
# Readmission by Diagnostic
# Calcula Day Of Readmission y CaseReadmission
# Particionado por PolicyId, MemberId, BillingProvider, Diagnostic
# Actualiza registros existentes + inserta los que faltan
# ============================================================
try:
    # ── Base: agrupación por Diagnostic (sin Procedure) ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ReadmissionDiagBase AS
        SELECT
            FC.ClaimHeaderId,
            Pol.PolicyId,
            FC.MemberId,
            FC.ClaimBillingProviderSkey,
            FC.ClaimPrimaryDiagnosticSkey,
            MIN(CAST(CONCAT(
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 1, 4), '-',
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 5, 2), '-',
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 7, 2)
            ) AS DATE)) AS ClaimServiceFromDate,
            MAX(FC.ClaimServiceToDate) AS ClaimServiceToDate
        FROM Gold.bgla_FactClaim FC
        INNER JOIN Gold.bgla_DimPolicy Pol ON FC.PolicySKey = CAST(Pol.PolicySKey AS STRING)
        INNER JOIN Gold.bgla_DimClaimReason dRS ON FC.ClaimReasonSKey = CAST(dRS.ClaimReasonSKey AS STRING)
        INNER JOIN Gold.bgla_DimStatus DS ON FC.ClaimStatusSKey = CAST(DS.StatusSKey AS STRING)
        WHERE FC.ControlVersion = 'Current Version'
          AND dRS.ClaimReasonName <> 'DELETED'
          AND DS.StatusTypeName = 'Claim Status'
          AND DS.StatusDescription = 'Processed'
          AND dRS.ClaimReasonName <> 'DENIED'
          AND FC.ClaimServiceFromDateKey > 19000101
        GROUP BY FC.ClaimHeaderId, Pol.PolicyId, FC.MemberId,
                 FC.ClaimBillingProviderSkey, FC.ClaimPrimaryDiagnosticSkey
    """)

    # ── Day of Readmission Diagnostic (LAG, remove negatives, recalculate) ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ReadmissionDiagDay AS
        SELECT ClaimHeaderId, ClaimServiceFromDate, ClaimServiceToDate,
               ClaimBillingProviderSkey, ClaimPrimaryDiagnosticSkey,
               PolicyId, MemberId,
               DATEDIFF(
                   ClaimServiceFromDate,
                   LAG(ClaimServiceToDate, 1) OVER (
                       PARTITION BY PolicyId, MemberId, ClaimBillingProviderSkey, ClaimPrimaryDiagnosticSkey
                       ORDER BY ClaimServiceFromDate, ClaimHeaderId
                   )
               ) AS DayOfReadmission
        FROM (
            SELECT ClaimHeaderId, ClaimServiceFromDate, ClaimServiceToDate,
                   ClaimBillingProviderSkey, ClaimPrimaryDiagnosticSkey,
                   PolicyId, MemberId
            FROM (
                SELECT *,
                    DATEDIFF(
                        ClaimServiceFromDate,
                        LAG(ClaimServiceToDate, 1) OVER (
                            PARTITION BY PolicyId, MemberId, ClaimBillingProviderSkey, ClaimPrimaryDiagnosticSkey
                            ORDER BY ClaimServiceFromDate, ClaimHeaderId
                        )
                    ) AS day_calc
                FROM vw_ReadmissionDiagBase
            ) t1
            WHERE day_calc IS NULL OR day_calc >= 0
        ) t2
    """)

    # ── CaseReadmission Diagnostic view ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_CaseReadmissionDiag AS
        SELECT
            RD.ClaimHeaderId,
            RD.ClaimServiceToDate,
            RD.ClaimServiceFromDate,
            RD.ClaimBillingProviderSkey,
            RD.PolicyId,
            RD.MemberId,
            RD.ClaimPrimaryDiagnosticSkey,
            RD.DayOfReadmission,
            CASE WHEN RD.DayOfReadmission BETWEEN 0 AND 30 THEN 1 ELSE 0 END AS CaseReadmission,
            Claims.ClaimPrimaryProcedureSkey,
            0 AS DayOfReadmissionProcedure,
            0 AS DayOfReadmissionProvider,
            0 AS CaseReadmissionProcedure,
            0 AS CaseReadmissionProvider,
            0 AS ReadmissionByProcedure,
            1 AS ReadmissionByDiagnostic,
            0 AS ReadmissionByProvider
        FROM vw_ReadmissionDiagDay RD
        INNER JOIN (
            SELECT ClaimHeaderId, MAX(ClaimPrimaryProcedureSkey) AS ClaimPrimaryProcedureSkey
            FROM Gold.bgla_FactClaim
            WHERE ControlVersion = 'Current Version'
            GROUP BY ClaimHeaderId
        ) Claims ON RD.ClaimHeaderId = Claims.ClaimHeaderId
        WHERE RD.DayOfReadmission IS NOT NULL
    """)

    # ── UPDATE existing rows with Diagnostic readmission ──
    spark.sql("""
        MERGE INTO Gold.bgla_FactClaimMemberCaseReadmission Fact
        USING vw_CaseReadmissionDiag ReadDiag
        ON ReadDiag.ClaimHeaderId = Fact.ClaimHeaderId
           AND ReadDiag.ClaimPrimaryDiagnosticSkey = Fact.ClaimPrimaryDiagnosticSkey
        WHEN MATCHED THEN UPDATE SET
            Fact.`Day Of Readmission` = ReadDiag.DayOfReadmission,
            Fact.CaseReadmission = ReadDiag.CaseReadmission,
            Fact.`Readmission by Diagnostic` = ReadDiag.ReadmissionByDiagnostic
    """)

    # ── INSERT new rows not already in the table ──
    spark.sql("""
        INSERT INTO Gold.bgla_FactClaimMemberCaseReadmission
        (ClaimHeaderId, ClaimServiceToDate, ClaimServiceFromDate, ClaimBillingProviderSkey,
         PolicyId, MemberId, ClaimPrimaryDiagnosticSkey,
         `Day Of Readmission`, CaseReadmission,
         ClaimPrimaryProcedureSkey, `Day Of Readmission Procedure`,
         `Day Of Readmission Provider`, CaseReadmissionProcedure, CaseReadmissionProvider,
         `Readmission by Procedure`, `Readmission by Diagnostic`, `Readmission by Provider`)
        SELECT
            RD.ClaimHeaderId, RD.ClaimServiceToDate, RD.ClaimServiceFromDate,
            RD.ClaimBillingProviderSkey, RD.PolicyId, RD.MemberId,
            RD.ClaimPrimaryDiagnosticSkey,
            RD.DayOfReadmission, RD.CaseReadmission,
            RD.ClaimPrimaryProcedureSkey, RD.DayOfReadmissionProcedure,
            RD.DayOfReadmissionProvider, RD.CaseReadmissionProcedure,
            RD.CaseReadmissionProvider, RD.ReadmissionByProcedure,
            RD.ReadmissionByDiagnostic, RD.ReadmissionByProvider
        FROM vw_CaseReadmissionDiag RD
        LEFT JOIN Gold.bgla_FactClaimMemberCaseReadmission Fact
            ON RD.ClaimHeaderId = Fact.ClaimHeaderId
        WHERE Fact.ClaimHeaderId IS NULL
    """)

    cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaimMemberCaseReadmission").first()["cnt"]
    print(f"[OK] Readmission by Diagnostic loaded/updated. Total rows: {cnt}")
except Exception as e:
    print(f"[WARN] Readmission by Diagnostic skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 46, Finished, Available, Finished, False)

[WARN] Readmission by Diagnostic skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Gold`.`bgla_FactClaimMemberCaseReadmission` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 19;
'MergeIntoTable (('ReadDiag.ClaimHeaderId = 'Fact.ClaimHeaderId) AND ('ReadDiag.ClaimPrimaryDiagnosticSkey = 'Fact.ClaimPrimaryDiagnosticSkey)), [updateaction(None, assignment('Fact.Day Of Readmission, 'ReadDiag.DayOfReadmission), assignment('Fact.CaseReadmission, 'ReadDiag.CaseReadmission), assignment('Fact.Readmission by Diagnostic, 'ReadDiag.ReadmissionByDiagnostic))]
:- 'SubqueryAlias Fact
:  +- 'UnresolvedRelation [Gold, bgla_FactClaimMemberCaseReadmission], [__required_write_privileges__=UPDATE], false
+- SubqueryAlias ReadDiag
   +- Subquery

In [44]:
# ============================================================
# Readmission by Provider
# Calcula Day Of Readmission Provider y CaseReadmissionProvider
# Particionado por PolicyId, MemberId, BillingProvider
# Actualiza registros existentes + inserta los que faltan
# ============================================================
try:
    # ── Base: agrupación por Provider (sin Diagnostic/Procedure) ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ReadmissionProvBase AS
        SELECT
            FC.ClaimHeaderId,
            Pol.PolicyId,
            FC.MemberId,
            FC.ClaimBillingProviderSkey,
            MIN(CAST(CONCAT(
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 1, 4), '-',
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 5, 2), '-',
                SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 7, 2)
            ) AS DATE)) AS ClaimServiceFromDate,
            MAX(FC.ClaimServiceToDate) AS ClaimServiceToDate
        FROM Gold.bgla_FactClaim FC
        INNER JOIN Gold.bgla_DimPolicy Pol ON FC.PolicySKey = CAST(Pol.PolicySKey AS STRING)
        INNER JOIN Gold.bgla_DimClaimReason dRS ON FC.ClaimReasonSKey = CAST(dRS.ClaimReasonSKey AS STRING)
        INNER JOIN Gold.bgla_DimStatus DS ON FC.ClaimStatusSKey = CAST(DS.StatusSKey AS STRING)
        WHERE FC.ControlVersion = 'Current Version'
          AND dRS.ClaimReasonName <> 'DELETED'
          AND DS.StatusTypeName = 'Claim Status'
          AND DS.StatusDescription = 'Processed'
          AND dRS.ClaimReasonName <> 'DENIED'
          AND FC.ClaimServiceFromDateKey > 19000101
        GROUP BY FC.ClaimHeaderId, Pol.PolicyId, FC.MemberId, FC.ClaimBillingProviderSkey
    """)

    # ── Day of Readmission Provider (LAG, remove negatives, recalculate) ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ReadmissionProvDay AS
        SELECT ClaimHeaderId, ClaimServiceFromDate, ClaimServiceToDate,
               ClaimBillingProviderSkey, PolicyId, MemberId,
               DATEDIFF(
                   ClaimServiceFromDate,
                   LAG(ClaimServiceToDate, 1) OVER (
                       PARTITION BY PolicyId, MemberId, ClaimBillingProviderSkey
                       ORDER BY ClaimServiceFromDate, ClaimHeaderId
                   )
               ) AS DayOfReadmissionProvider
        FROM (
            SELECT ClaimHeaderId, ClaimServiceFromDate, ClaimServiceToDate,
                   ClaimBillingProviderSkey, PolicyId, MemberId
            FROM (
                SELECT *,
                    DATEDIFF(
                        ClaimServiceFromDate,
                        LAG(ClaimServiceToDate, 1) OVER (
                            PARTITION BY PolicyId, MemberId, ClaimBillingProviderSkey
                            ORDER BY ClaimServiceFromDate, ClaimHeaderId
                        )
                    ) AS day_calc
                FROM vw_ReadmissionProvBase
            ) t1
            WHERE day_calc IS NULL OR day_calc >= 0
        ) t2
    """)

    # ── CaseReadmission Provider view ──
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_CaseReadmissionProv AS
        SELECT
            RP.ClaimHeaderId,
            RP.ClaimServiceToDate,
            RP.ClaimServiceFromDate,
            RP.ClaimBillingProviderSkey,
            RP.PolicyId,
            RP.MemberId,
            Claims.ClaimPrimaryDiagnosticSkey,
            0 AS DayOfReadmission,
            0 AS CaseReadmission,
            Claims.ClaimPrimaryProcedureSkey,
            0 AS DayOfReadmissionProcedure,
            RP.DayOfReadmissionProvider,
            0 AS CaseReadmissionProcedure,
            CASE WHEN RP.DayOfReadmissionProvider BETWEEN 0 AND 30 THEN 1 ELSE 0 END AS CaseReadmissionProvider,
            0 AS ReadmissionByProcedure,
            0 AS ReadmissionByDiagnostic,
            1 AS ReadmissionByProvider
        FROM vw_ReadmissionProvDay RP
        INNER JOIN (
            SELECT ClaimHeaderId,
                   MAX(ClaimPrimaryDiagnosticSkey) AS ClaimPrimaryDiagnosticSkey,
                   MAX(ClaimPrimaryProcedureSkey) AS ClaimPrimaryProcedureSkey
            FROM Gold.bgla_FactClaim
            WHERE ControlVersion = 'Current Version'
            GROUP BY ClaimHeaderId
        ) Claims ON RP.ClaimHeaderId = Claims.ClaimHeaderId
        WHERE RP.DayOfReadmissionProvider IS NOT NULL
    """)

    # ── UPDATE existing rows with Provider readmission ──
    spark.sql("""
        MERGE INTO Gold.bgla_FactClaimMemberCaseReadmission Fact
        USING vw_CaseReadmissionProv ReadProv
        ON ReadProv.ClaimHeaderId = Fact.ClaimHeaderId
           AND ReadProv.ClaimBillingProviderSkey = Fact.ClaimBillingProviderSkey
        WHEN MATCHED THEN UPDATE SET
            Fact.`Day Of Readmission Provider` = ReadProv.DayOfReadmissionProvider,
            Fact.CaseReadmissionProvider = ReadProv.CaseReadmissionProvider,
            Fact.`Readmission by Provider` = ReadProv.ReadmissionByProvider
    """)

    # ── INSERT new rows not already in the table ──
    spark.sql("""
        INSERT INTO Gold.bgla_FactClaimMemberCaseReadmission
        (ClaimHeaderId, ClaimServiceToDate, ClaimServiceFromDate, ClaimBillingProviderSkey,
         PolicyId, MemberId, ClaimPrimaryDiagnosticSkey,
         `Day Of Readmission`, CaseReadmission,
         ClaimPrimaryProcedureSkey, `Day Of Readmission Procedure`,
         `Day Of Readmission Provider`, CaseReadmissionProcedure, CaseReadmissionProvider,
         `Readmission by Procedure`, `Readmission by Diagnostic`, `Readmission by Provider`)
        SELECT
            RP.ClaimHeaderId, RP.ClaimServiceToDate, RP.ClaimServiceFromDate,
            RP.ClaimBillingProviderSkey, RP.PolicyId, RP.MemberId,
            RP.ClaimPrimaryDiagnosticSkey,
            RP.DayOfReadmission, RP.CaseReadmission,
            RP.ClaimPrimaryProcedureSkey, RP.DayOfReadmissionProcedure,
            RP.DayOfReadmissionProvider, RP.CaseReadmissionProcedure,
            RP.CaseReadmissionProvider, RP.ReadmissionByProcedure,
            RP.ReadmissionByDiagnostic, RP.ReadmissionByProvider
        FROM vw_CaseReadmissionProv RP
        LEFT JOIN Gold.bgla_FactClaimMemberCaseReadmission Fact
            ON RP.ClaimHeaderId = Fact.ClaimHeaderId
        WHERE Fact.ClaimHeaderId IS NULL
    """)

    cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaimMemberCaseReadmission").first()["cnt"]
    print(f"[OK] Readmission by Provider loaded/updated. Total rows: {cnt}")
except Exception as e:
    print(f"[WARN] Readmission by Provider skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 47, Finished, Available, Finished, False)

[WARN] Readmission by Provider skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Gold`.`bgla_FactClaimMemberCaseReadmission` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 19;
'MergeIntoTable (('ReadProv.ClaimHeaderId = 'Fact.ClaimHeaderId) AND ('ReadProv.ClaimBillingProviderSkey = 'Fact.ClaimBillingProviderSkey)), [updateaction(None, assignment('Fact.Day Of Readmission Provider, 'ReadProv.DayOfReadmissionProvider), assignment('Fact.CaseReadmissionProvider, 'ReadProv.CaseReadmissionProvider), assignment('Fact.Readmission by Provider, 'ReadProv.ReadmissionByProvider))]
:- 'SubqueryAlias Fact
:  +- 'UnresolvedRelation [Gold, bgla_FactClaimMemberCaseReadmission], [__required_write_privileges__=UPDATE], false
+- SubqueryAlias 

In [45]:
# ============================================================
# Hospital_Emergency cross-match
# Para reclamos que pasen por ServiceIndicator de Hospitalización
# y que tengan otra línea de Emergencia dentro de 2 días
# Emergency IDs: 197,198,679,708,720,769
# Hospitalization IDs: 316,344,346,348,349,350,351,352,353,354,355,619,620,650,651,653,654,709
# ============================================================
try:
    spark.sql("""
        MERGE INTO Gold.bgla_FactClaim FactClaim
        USING (
            SELECT DISTINCT
                h.ClaimHeaderId AS HeaderId_Hospitalization,
                HE.PolicyId,
                HE.MemberId
            FROM (
                -- Emergency claims
                SELECT
                    FC.ClaimHeaderId,
                    Pol.PolicyId,
                    FC.MemberId,
                    CAST(CONCAT(
                        SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 1, 4), '-',
                        SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 5, 2), '-',
                        SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 7, 2)
                    ) AS DATE) AS ClaimServiceFromDate,
                    FC.ClaimServiceToDate
                FROM Gold.bgla_FactClaim FC
                INNER JOIN Gold.bgla_DimPolicy Pol ON FC.PolicySKey = CAST(Pol.PolicySKey AS STRING)
                INNER JOIN Gold.bgla_DimClaimReason dRS ON FC.ClaimReasonSKey = CAST(dRS.ClaimReasonSKey AS STRING)
                INNER JOIN Gold.bgla_DimServiceIndicator SInd
                    ON CAST(SInd.ServiceIndicatorSKey AS STRING) = FC.ClaimLineDetailServiceindicatorSkey
                WHERE dRS.ClaimReasonName <> 'DELETED'
                  AND FC.ClaimServiceFromDateKey > 19000101
                  AND SInd.ServiceIndicatorId IN (197, 198, 679, 708, 720, 769)
            ) HE
            INNER JOIN (
                -- Hospitalization claims
                SELECT
                    FC.ClaimHeaderId,
                    Pol.PolicyId,
                    FC.MemberId,
                    CAST(CONCAT(
                        SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 1, 4), '-',
                        SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 5, 2), '-',
                        SUBSTRING(CAST(FC.ClaimServiceFromDateKey AS STRING), 7, 2)
                    ) AS DATE) AS ClaimServiceFromDate,
                    FC.ClaimServiceToDate
                FROM Gold.bgla_FactClaim FC
                INNER JOIN Gold.bgla_DimPolicy Pol ON FC.PolicySKey = CAST(Pol.PolicySKey AS STRING)
                INNER JOIN Gold.bgla_DimClaimReason dRS ON FC.ClaimReasonSKey = CAST(dRS.ClaimReasonSKey AS STRING)
                INNER JOIN Gold.bgla_DimServiceIndicator SInd
                    ON CAST(SInd.ServiceIndicatorSKey AS STRING) = FC.ClaimLineDetailServiceindicatorSkey
                WHERE dRS.ClaimReasonName <> 'DELETED'
                  AND FC.ClaimServiceFromDateKey > 19000101
                  AND SInd.ServiceIndicatorId IN (316, 344, 346, 348, 349, 350, 351, 352, 353, 354, 355,
                                                   619, 620, 650, 651, 653, 654, 709)
            ) h
            ON HE.PolicyId = h.PolicyId
               AND HE.MemberId = h.MemberId
               AND h.ClaimServiceFromDate BETWEEN HE.ClaimServiceToDate AND DATE_ADD(HE.ClaimServiceToDate, 2)
        ) HospEmerg
        ON FactClaim.ClaimHeaderId = HospEmerg.HeaderId_Hospitalization
           AND FactClaim.PolicyId = HospEmerg.PolicyId
           AND FactClaim.MemberId = HospEmerg.MemberId
        WHEN MATCHED THEN UPDATE SET FactClaim.IsHospital_Emergency = 1
    """)
    print("[OK] Hospital_Emergency cross-match applied")
except Exception as e:
    print(f"[WARN] Hospital_Emergency skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 48, Finished, Available, Finished, False)

[OK] Hospital_Emergency cross-match applied


In [46]:
# ============================================================
# Orphan Deletion
# Elimina registros de Gold.bgla_FactClaim cuyos IDs fuente
# ya no existen en las tablas Bronze
# Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims (DELETE orphans)
# ============================================================

# ── Orphan ClaimLineDetail ──
try:
    deleted_line = spark.sql("""
        DELETE FROM Gold.bgla_FactClaim
        WHERE ClaimLineDetailId <> -101
          AND NOT EXISTS (
              SELECT 1 FROM Bronze.AmigosPlus_AMP_Claim_LineDetail src
              WHERE src.ClaimLineDetailId = Gold.bgla_FactClaim.ClaimLineDetailId
          )
    """)
    print("[OK] Orphan ClaimLineDetail rows deleted")
except Exception as e:
    print(f"[WARN] Orphan ClaimLineDetail deletion skipped: {e}")

# ── Orphan ClaimDetail ──
try:
    spark.sql("""
        DELETE FROM Gold.bgla_FactClaim
        WHERE NOT EXISTS (
            SELECT 1 FROM Bronze.AmigosPlus_AMP_Claim_Detail src
            WHERE src.ClaimDetailId = Gold.bgla_FactClaim.ClaimDetailId
        )
    """)
    print("[OK] Orphan ClaimDetail rows deleted")
except Exception as e:
    print(f"[WARN] Orphan ClaimDetail deletion skipped: {e}")

# ── Orphan ClaimHeader ──
try:
    spark.sql("""
        DELETE FROM Gold.bgla_FactClaim
        WHERE NOT EXISTS (
            SELECT 1 FROM Bronze.AmigosPlus_AMP_Claim_Header src
            WHERE src.HeaderId = Gold.bgla_FactClaim.ClaimHeaderId
        )
    """)
    print("[OK] Orphan ClaimHeader rows deleted")
except Exception as e:
    print(f"[WARN] Orphan ClaimHeader deletion skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 49, Finished, Available, Finished, False)

[WARN] Orphan ClaimLineDetail deletion skipped: [DELTA_UNSUPPORTED_SUBQUERY] Subqueries are not supported in the DELETE (condition = ((NOT (spark_catalog.Gold.bgla_FactClaim.ClaimLineDetailId = CAST(-101 AS BIGINT))) AND (NOT exists(spark_catalog.Gold.bgla_FactClaim.ClaimLineDetailId)))).
[WARN] Orphan ClaimDetail deletion skipped: [DELTA_UNSUPPORTED_SUBQUERY] Subqueries are not supported in the DELETE (condition = (NOT exists(spark_catalog.Gold.bgla_FactClaim.ClaimDetailId))).
[WARN] Orphan ClaimHeader deletion skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Bronze`.`AmigosPlus_AMP_Claim_Header` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 4 pos 26;
'DeltaDelete NOT exists#151428 []
:  +- 'Project [unresolvedalias(1, None)]

In [47]:
# ============================================================
# FirstClaim Update
# Marca FirstClaim = 1 para el reclamo más antiguo por PolicyId
# (filtrado por ClaimHeaderIds afectados en las tablas staging)
# Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims #Claim + UPDATE
# Silver.TmpFactClaimAmount = ETLStaging.dbo.__TMPvwTMPBGLAEDW___ClaimCostData
# Silver.TmpFactClaim       = ETLStaging.dbo.__TMPvwTMPBGLAEDW___FactClaim
# ============================================================
try:
    # Construir vista con ROW_NUMBER por PolicyId (solo headers afectados)
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_FirstClaimUpdate AS
        SELECT DISTINCT PolicyId, ClaimHeaderId, ClaimServiceFromDateKey,
            ROW_NUMBER() OVER (PARTITION BY PolicyId ORDER BY ClaimServiceFromDateKey ASC) AS olddate
        FROM Gold.bgla_FactClaim
        WHERE ClaimHeaderId IN (
            SELECT DISTINCT ClaimHeaderId FROM Silver.TmpFactClaimAmount
            UNION
            SELECT DISTINCT HeaderId AS ClaimHeaderId FROM Silver.TmpFactClaim
        )
    """)

    # Actualizar FirstClaim = 1 para el primer reclamo de cada póliza
    spark.sql("""
        MERGE INTO Gold.bgla_FactClaim a
        USING (
            SELECT ClaimHeaderId, ClaimServiceFromDateKey
            FROM vw_FirstClaimUpdate
            WHERE olddate = 1
        ) b
        ON a.ClaimHeaderId = b.ClaimHeaderId
           AND a.ClaimServiceFromDateKey = b.ClaimServiceFromDateKey
        WHEN MATCHED THEN UPDATE SET a.FirstClaim = 1
    """)
    print("[OK] FirstClaim update applied")
except Exception as e:
    print(f"[WARN] FirstClaim update skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 50, Finished, Available, Finished, False)

[WARN] FirstClaim update skipped: [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE] Cannot perform Merge as multiple source rows matched and attempted to modify the same
target row in the Delta table in possibly conflicting ways. By SQL semantics of Merge,
when multiple source rows match on the same target row, the result may be ambiguous
as it is unclear which source row should be used to update or delete the matching
target row. You can preprocess the source table to eliminate the possibility of
multiple matches. Please refer to
https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge


In [48]:
# ============================================================
# ControlVersion Update on FactClaimPayment
# Recalcula ControlVersion con DENSE_RANK por ClaimHeaderId
# Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims (lines 72-80)
# Tabla: Gold.bgla_FactClaimPayment
# ============================================================
try:
    if spark.catalog.tableExists("Gold.bgla_FactClaimPayment"):
        spark.sql("""
            MERGE INTO Gold.bgla_FactClaimPayment t
            USING (
                SELECT FactClaimPaymentSKey,
                    DENSE_RANK() OVER (PARTITION BY ClaimHeaderId ORDER BY ClaimDetailId DESC) AS ControlVersionNew
                FROM Gold.bgla_FactClaimPayment
            ) tt
            ON t.FactClaimPaymentSKey = tt.FactClaimPaymentSKey
               AND t.ControlVersion <> CAST(tt.ControlVersionNew AS STRING)
            WHEN MATCHED THEN UPDATE SET t.ControlVersion = CAST(tt.ControlVersionNew AS STRING)
        """)
        print("[OK] ControlVersion updated on FactClaimPayment")
    else:
        print("[INFO] Gold.bgla_FactClaimPayment does not exist yet — skipping ControlVersion update")
except Exception as e:
    print(f"[WARN] ControlVersion update on FactClaimPayment skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 51, Finished, Available, Finished, False)

[OK] ControlVersion updated on FactClaimPayment


In [49]:
# ============================================================
# ClaimValidation Update
# Marca reclamos como 'CLOSED With PAID' o 'DENIED With PAID'
# cuando un estado CLOSED/DENIED es seguido por un PAID
# (filtrado por ClaimHeaderIds afectados en las tablas staging)
# Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims ClaimValidation
# Silver.TmpFactClaimAmount = ETLStaging.dbo.__TMPvwTMPBGLAEDW___ClaimCostData
# Silver.TmpFactClaim       = ETLStaging.dbo.__TMPvwTMPBGLAEDW___FactClaim
# ============================================================
try:
    # Vista de headers afectados
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_AffectedHeaders AS
        SELECT DISTINCT ClaimHeaderId FROM Silver.TmpFactClaimAmount
        UNION
        SELECT DISTINCT HeaderId AS ClaimHeaderId FROM Silver.TmpFactClaim
    """)

    # CTE: CLOSED/DENIED seguidos de PAID (LogId mayor = evento posterior)
    spark.sql("""
        CREATE OR REPLACE TEMPORARY VIEW vw_ClaimValidation AS
        SELECT DISTINCT c.ClaimHeaderId, c.Reason
        FROM (
            -- Reclamos cerrados/denegados
            SELECT DISTINCT tat.ClaimHeaderId, tat.LogId, tat.Reason
            FROM Gold.bgla_FactClaimTAT tat
            INNER JOIN vw_AffectedHeaders ah ON tat.ClaimHeaderId = ah.ClaimHeaderId
            WHERE tat.Status = 'P'
              AND tat.Reason IN ('CLOSED', 'DENIED')
              AND tat.Iterations <> 0
        ) c
        INNER JOIN (
            -- Reclamos pagados posteriormente
            SELECT tat.ClaimHeaderId, tat.LogId
            FROM Gold.bgla_FactClaimTAT tat
            INNER JOIN vw_AffectedHeaders ah ON tat.ClaimHeaderId = ah.ClaimHeaderId
            WHERE tat.Status = 'P'
              AND tat.Reason IN ('SMR', 'SBANk', 'PAID')
              AND tat.Iterations <> 0
        ) a
        ON c.ClaimHeaderId = a.ClaimHeaderId AND a.LogId > c.LogId
    """)

    # Actualizar ClaimValidation en Gold
    spark.sql("""
        MERGE INTO Gold.bgla_FactClaim fc
        USING vw_ClaimValidation cv
        ON fc.ClaimHeaderId = cv.ClaimHeaderId
        WHEN MATCHED THEN UPDATE SET
            fc.ClaimValidation = CASE
                WHEN cv.Reason = 'CLOSED' THEN 'CLOSED With PAID'
                WHEN cv.Reason = 'DENIED' THEN 'DENIED With PAID'
                ELSE '_Not defined'
            END
    """)
    print("[OK] ClaimValidation update applied")
except Exception as e:
    print(f"[WARN] ClaimValidation update skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 52, Finished, Available, Finished, False)

[WARN] ClaimValidation update skipped: [TABLE_OR_VIEW_NOT_FOUND] The table or view `Gold`.`bgla_FactClaimTAT` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 7 pos 17;
'CreateViewCommand `vw_ClaimValidation`, SELECT DISTINCT c.ClaimHeaderId, c.Reason
        FROM (
            -- Reclamos cerrados/denegados
            SELECT DISTINCT tat.ClaimHeaderId, tat.LogId, tat.Reason
            FROM Gold.bgla_FactClaimTAT tat
            INNER JOIN vw_AffectedHeaders ah ON tat.ClaimHeaderId = ah.ClaimHeaderId
            WHERE tat.Status = 'P'
              AND tat.Reason IN ('CLOSED', 'DENIED')
              AND tat.Iterations <> 0
        ) c
        INNER JOIN (
            -- Reclamos pagados posteriormente
            SELECT tat.ClaimHe

In [50]:
# ============================================================
# Stale Data Cleanup — Tablas relacionadas
# Elimina registros anteriores al día/carga actual
# Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims (lines 143-160)
# Tablas: Gold.bgla_FactScheduleLimits, Gold.bgla_FactMemberAccumulator,
#          Gold.bgla_FactClaimReferral, Gold.bgla_FactClaimSubLine
# ============================================================

# ── FactScheduleLimits: DELETE WHERE LoadDate < today ──
try:
    if spark.catalog.tableExists("Gold.bgla_FactScheduleLimits"):
        spark.sql("DELETE FROM Gold.bgla_FactScheduleLimits WHERE CAST(LoadDate AS DATE) < current_date()")
        cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactScheduleLimits").first()["cnt"]
        print(f"[OK] FactScheduleLimits stale rows deleted. Remaining: {cnt}")
    else:
        print("[INFO] Gold.bgla_FactScheduleLimits does not exist yet — skipping")
except Exception as e:
    print(f"[WARN] FactScheduleLimits cleanup skipped: {e}")

# ── FactMemberAccumulator: DELETE WHERE LoadDate < MAX(LoadDate) ──
try:
    if spark.catalog.tableExists("Gold.bgla_FactMemberAccumulator"):
        max_load = spark.sql("SELECT MAX(CAST(LoadDate AS DATE)) AS mld FROM Gold.bgla_FactMemberAccumulator").first()["mld"]
        if max_load is not None:
            spark.sql(f"DELETE FROM Gold.bgla_FactMemberAccumulator WHERE CAST(LoadDate AS DATE) < DATE '{max_load}'")
        cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactMemberAccumulator").first()["cnt"]
        print(f"[OK] FactMemberAccumulator stale rows deleted. Remaining: {cnt}")
    else:
        print("[INFO] Gold.bgla_FactMemberAccumulator does not exist yet — skipping")
except Exception as e:
    print(f"[WARN] FactMemberAccumulator cleanup skipped: {e}")

# ── FactClaimReferral: DELETE WHERE LoadDate < today ──
try:
    if spark.catalog.tableExists("Gold.bgla_FactClaimReferral"):
        spark.sql("DELETE FROM Gold.bgla_FactClaimReferral WHERE CAST(LoadDate AS DATE) < current_date()")
        cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaimReferral").first()["cnt"]
        print(f"[OK] FactClaimReferral stale rows deleted. Remaining: {cnt}")
    else:
        print("[INFO] Gold.bgla_FactClaimReferral does not exist yet — skipping")
except Exception as e:
    print(f"[WARN] FactClaimReferral cleanup skipped: {e}")

# ── FactClaimSubLine: DELETE WHERE LoadDate < today ──
try:
    if spark.catalog.tableExists("Gold.bgla_FactClaimSubLine"):
        spark.sql("DELETE FROM Gold.bgla_FactClaimSubLine WHERE CAST(LoadDate AS DATE) < current_date()")
        cnt = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaimSubLine").first()["cnt"]
        print(f"[OK] FactClaimSubLine stale rows deleted. Remaining: {cnt}")
    else:
        print("[INFO] Gold.bgla_FactClaimSubLine does not exist yet — skipping")
except Exception as e:
    print(f"[WARN] FactClaimSubLine cleanup skipped: {e}")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 53, Finished, Available, Finished, False)

[OK] FactScheduleLimits stale rows deleted. Remaining: 0
[INFO] Gold.bgla_FactMemberAccumulator does not exist yet — skipping
[INFO] Gold.bgla_FactClaimReferral does not exist yet — skipping
[OK] FactClaimSubLine stale rows deleted. Remaining: 0


In [51]:
# ============================================================
# CLEANUP + OPTIMIZE + UNCACHE
# 1. Limpiar staging (DELETE Silver.TmpFactClaim)
# 2. DELETE non-actual rows (ActualRowFlag = 0)
#    Equivalente: usp_DataPrepare_BGLADW_ActualRowFlagClaims (line 44)
# 3. OPTIMIZE Gold table (V-Order)
# 4. UNCACHE all temp views
# ============================================================

# ── Step 1: Limpiar staging (no DROP, se mantiene la tabla) ──
if spark.catalog.tableExists("Silver.TmpFactClaim"):
    spark.sql("DELETE FROM Silver.TmpFactClaim")
    print("[OK] Silver.TmpFactClaim truncated (rows deleted)")

# ── Step 2: Delete non-actual rows (usp_DataPrepare_BGLADW_ActualRowFlagClaims L44) ──
spark.sql("DELETE FROM Gold.bgla_FactClaim WHERE ActualRowFlag = 0")
print("[OK] Non-actual rows deleted")

# ── Step 3: OPTIMIZE with V-Order ──
spark.sql("OPTIMIZE Gold.bgla_FactClaim VORDER")
print("[OK] Gold.bgla_FactClaim optimized with V-Order")

# ── Step 4: UNCACHE all temp views ──
views_to_uncache = [
    "vw_TClaimDetail", "vw_ClaimLog", "vw_CurrentVersion", "vw_ProcessedDate",
    "vw_DiagnosticLine", "vw_ProcedureLine", "vw_ClaimSubmission",
    "vw_TmpReceived", "vw_TmpServices", "vw_PolicyBaseCurrency",
    "vw_Payee", "vw_ReceivedMethodDigital", "vw_LocalTeamTAT",
    "vw_HighCostClaim", "vw_CoinsurancePercent", "vw_ClearingHouse",
    "vw_SubLineDetailValue", "vw_DetailOverride", "vw_OverrideMin",
    "vw_OverrideReason", "vw_PostedDate", "vw_Coverage",
    "vwTmp_RegistrosFactClaim",
    "vw_FirstClaimUpdate", "vw_AffectedHeaders", "vw_ClaimValidation",
    "vw_ReadmissionProcedureBase", "vw_ReadmissionProcedureDay",
    "vw_ReadmissionDiagBase", "vw_ReadmissionDiagDay", "vw_CaseReadmissionDiag",
    "vw_ReadmissionProvBase", "vw_ReadmissionProvDay", "vw_CaseReadmissionProv",
    "vw_DimStatus_bc", "vw_DimClaimReason_bc", "vw_DimDate_bc",
    "vw_DimPolicyMember_bc", "vw_DimProvider_bc", "vw_DimSchedule_bc",
    "vw_DimServiceNetwork_bc", "vw_DimServiceIndicator_bc",
    "vw_DimDiagnostic_bc", "vw_DimProcedure_bc", "vw_DimPlaceOfService_bc",
    "vw_DimTypeOfService_bc", "vw_DimPriority_bc", "vw_DimGeography_bc",
    "vw_DimAccumulatorScope_bc", "vw_DimBillType_bc", "vw_DimPayTo_bc",
    "vw_DimPayee_bc", "vw_DimPolicy_bc", "vw_DimCurrency_bc",
    "vw_DimReceivedMethod_bc"
]

for v in views_to_uncache:
    try:
        spark.catalog.uncacheTable(v)
    except Exception:
        pass

spark.catalog.clearCache()
print("[OK] All caches cleared")

final_count = spark.sql("SELECT COUNT(*) AS cnt FROM Gold.bgla_FactClaim").first()["cnt"]
print(f"[DONE] nb_FactClaims completed. Gold.bgla_FactClaim: {final_count} rows")

StatementMeta(, ff8b112a-c668-4b80-8dce-173961b9b430, 54, Finished, Available, Finished, False)

[OK] Silver.TmpFactClaim truncated (rows deleted)
[OK] Non-actual rows deleted
[OK] Gold.bgla_FactClaim optimized with V-Order
[OK] All caches cleared
[DONE] nb_FactClaims completed. Gold.bgla_FactClaim: 12945318 rows
